# SDG pipeline — Qwen3 32B (NVFP4, thinking disabled)

Stage 1 + Stage 2 LLM scoring, the seven SDG–SDG matrices (A–G) with
signed-gated propagation, baselines B1–B4, and the per-document methods
(`doc_neighbours`, `doc_entities`). Writes raw LLM probabilities to the
shared cache that `03_compare_ablate.ipynb` consumes.

Backbone is selected by `ACTIVE_MODEL` in the config cell. Paths come
from the root resolved in the Paths cell — see README section 1.


In [ ]:
!pip -q install --upgrade vllm openai sentence-transformers zenodo-get scikit-learn "pandas==2.2.2" scipy "requests==2.32.4" "numba<0.62.0" "opentelemetry-api<1.39.0" "opentelemetry-sdk<1.39.0" "jedi>=0.16"
# vllm >= 0.7 needed for Gemma 4 architecture
# openai is the OpenAI-compatible client; vLLM exposes that API
# sentence-transformers for SBERT (Matrix C)

## Notebook autosave (run once, before Stage 1+2)
Saves a timestamped per-model snapshot to Drive every 15 minutes.

In [ ]:
import os, json, gc, time, shutil, re, random, requests, io, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, hamming_loss
from openai import OpenAI                  # vLLM exposes an OpenAI-compatible API
import warnings; warnings.filterwarnings('ignore')

# ── Pipeline config ──────────────────────────────────────────────────────────
N_FEWSHOT   = 40
N_VAL       = 200
N_TEST      = 10000     # for full run; lower for smoke testing
BATCH_SIZE  = 5
CONCURRENCY = 64        # vLLM batches efficiently — much higher than Claude's 6
ALPHA_GRID  = [0.7, 0.8, 0.85, 0.9, 0.95]
CONF_GRID   = [0.5, 0.6, 0.7]
STAGE2_THRESHOLD = 0.3
SEED        = 42
# VERSION is auto-derived from N_TEST so caches never collide across scales.
# Override only if you want to fork a separate run (e.g. with seed change).
VERSION     = f'v5_multi_{ACTIVE_MODEL}_n{N_TEST}'   # per-model cache (v5 bump for matrix-A fix)
MATRIX_VERSION = 'v3_2'    # which Pradhan/Warchold matrix file to load (v5: explicit, no os.walk lottery)
random.seed(SEED); np.random.seed(SEED)

# ── Smoke-test toggle (uncomment one block to override the values above) ─────
# 100-doc smoke (~10 min wall time):
# N_FEWSHOT, N_VAL, N_TEST = 10, 30, 100   # then VERSION auto-becomes 'local_v3_n100'
# 2.5k smoke (~1 hour wall time):
# N_FEWSHOT, N_VAL, N_TEST = 40, 200, 2500  # then VERSION auto-becomes 'local_v3_n2500'

# ════════════════════════════════════════════════════════════════════════════
# MODEL REGISTRY — switch by changing ACTIVE_MODEL below.
# ════════════════════════════════════════════════════════════════════════════
# All models in this registry are 24-32B class on NVFP4 (Blackwell fast-path).
# Per-model cache lives at SHARED_LLM_CACHE_DIR/<key>/, so you can run all
# four back-to-back in one Colab session without stomping each other.
# ════════════════════════════════════════════════════════════════════════════

MODELS = {
    "gemma4-26b": {
        "hf_id":          "bg-digitalservices/Gemma-4-26B-A4B-it-NVFP4",
        "quantization":   "modelopt",
        "moe_backend":    "marlin",
        "kv_cache_dtype": "fp8",
        "max_model_len":  8192,
        "gpu_mem_frac":   0.85,
        "needs_nvfp4_patch": True,
        "extra_env":      {"VLLM_NVFP4_GEMM_BACKEND": "marlin"},
        "extra_args":     [],
        "is_reasoning":      False,
        "strip_think_blocks": False,
        "chat_template_kwargs": None,
    },
    "mistral-24b": {
        "hf_id":          "RedHatAI/Mistral-Small-3.2-24B-Instruct-2506-NVFP4",
        "quantization":   "compressed-tensors",
        "moe_backend":    None,
        "kv_cache_dtype": "fp8",
        "max_model_len":  8192,
        "gpu_mem_frac":   0.88,
        "needs_nvfp4_patch": False,
        "extra_env":      {},
        "extra_args":     ["--tokenizer_mode", "mistral"],
        "is_reasoning":      False,
        "strip_think_blocks": False,
        "chat_template_kwargs": None,
    },
    "qwen3-32b": {
        "hf_id":          "RedHatAI/Qwen3-32B-NVFP4",
        "quantization":   "compressed-tensors",
        "moe_backend":    None,
        "kv_cache_dtype": "fp8",
        "max_model_len":  8192,
        "gpu_mem_frac":   0.88,
        "needs_nvfp4_patch": False,
        "extra_env":      {},
        "extra_args":     ["--reasoning-parser", "qwen3"],
        "is_reasoning":      True,
        "strip_think_blocks": False,
        "chat_template_kwargs": {"enable_thinking": False},
    },
    "mixtral-8x7b": {
        "hf_id":          "RedHatAI/Mixtral-8x7B-Instruct-v0.1-AutoFP8",
        "quantization":   "fp8",
        "moe_backend":    None,            # FP8 MoE handled natively by vLLM
        "kv_cache_dtype": "fp8",
        "max_model_len":  8192,
        "gpu_mem_frac":   0.92,            # FP8 weights are ~45 GB, tighter
        "needs_nvfp4_patch": False,
        "extra_env":      {},
        "extra_args":     [],
        "is_reasoning":      False,
        "strip_think_blocks": False,
        "chat_template_kwargs": None,
    },
}

ACTIVE_MODEL = "qwen3-32b"     # ← change this one line to switch models

# Resolve the chosen model from the registry — these names are exactly
# what the rest of the notebook uses (so v3.2/v4 cells need no further changes).
assert ACTIVE_MODEL in MODELS, f"ACTIVE_MODEL={ACTIVE_MODEL!r} not in MODELS keys: {list(MODELS)}"
_CFG          = MODELS[ACTIVE_MODEL]
MODEL_NAME    = _CFG["hf_id"]
QUANTIZATION  = _CFG["quantization"]
MOE_BACKEND   = _CFG["moe_backend"]
KV_CACHE_DTYPE = _CFG["kv_cache_dtype"]
GPU_MEM_FRAC  = _CFG["gpu_mem_frac"]
MAX_MODEL_LEN = _CFG["max_model_len"]
TENSOR_PARALLEL = 1
DTYPE         = "auto"

# Reasoning-mode handling: thinking OFF by default. Flip to True for
# models with a built-in toggle (e.g. qwen3-32b).
THINKING_MODE = False

FORCE_REFRESH = False

VLLM_PORT     = 8000
VLLM_BASE_URL = f'http://localhost:{VLLM_PORT}/v1'

SDG_NAMES = {
    1:'No Poverty', 2:'Zero Hunger', 3:'Good Health', 4:'Quality Education',
    5:'Gender Equality', 6:'Clean Water', 7:'Clean Energy', 8:'Decent Work',
    9:'Innovation', 10:'Reduced Inequalities', 11:'Sustainable Cities',
    12:'Responsible Consumption', 13:'Climate Action', 14:'Life Below Water',
    15:'Life on Land', 16:'Peace & Justice', 17:'Partnerships'
}
NUM_GOALS = 17
print(f'Config: N_TEST={N_TEST}  CONCURRENCY={CONCURRENCY}  VERSION={VERSION}')
print(f'LLM backend [{ACTIVE_MODEL}]: {MODEL_NAME}  (quant={QUANTIZATION})')
print(f'Available models in registry: {list(MODELS)}')


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# All paths derive from one root: $SDG_ROOT, else config.yaml, else Drive on
# Colab, else ./sdg_data. See README section 1.
import os, sys

def _resolve_sdg_root():
    r = os.environ.get('SDG_ROOT')
    if r:
        return os.path.abspath(os.path.expanduser(r))
    for cand in ('config.yaml', os.path.join('..', 'config.yaml')):
        if os.path.exists(cand):
            with open(cand) as fh:
                for line in fh:
                    line = line.split('#')[0].strip()
                    if line.startswith('root:'):
                        v = line.split(':', 1)[1].strip().strip('"\'')
                        if v:
                            return os.path.abspath(os.path.expanduser(v))
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            return '/content/drive/MyDrive/sdg-llm-graph'
        except Exception:
            pass
    return os.path.abspath('./sdg_data')

SDG_ROOT   = _resolve_sdg_root()
GRAPH_DIR  = os.path.join(SDG_ROOT, 'sdggraph')
DRIVE_ROOT = os.path.join(SDG_ROOT, 'aurora_sdg_graph_full')
DATA_CACHE = os.path.join(DRIVE_ROOT, 'data_cache')
for _d in (SDG_ROOT, GRAPH_DIR, DRIVE_ROOT, DATA_CACHE):
    os.makedirs(_d, exist_ok=True)

print(f'SDG_ROOT   : {SDG_ROOT}')
print(f'GRAPH_DIR  : {GRAPH_DIR}')
print(f'DATA_CACHE : {DATA_CACHE}')

# Per-run output dir and per-model LLM cache (layout unchanged from the runs
# reported in the paper; only the root above moves).
RESULTS_DIR          = os.path.join(DRIVE_ROOT, f'results_{VERSION}')
LOCAL_MATRIX         = os.path.join(GRAPH_DIR, f'sdg_interaction_matrix_{MATRIX_VERSION}.json')
SHARED_LLM_CACHE_DIR = os.path.join(DRIVE_ROOT, 'shared_llm_cache_local', ACTIVE_MODEL)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SHARED_LLM_CACHE_DIR, exist_ok=True)
print(f'RESULTS_DIR: {RESULTS_DIR}')
print(f'SHARED_LLM_CACHE_DIR: {SHARED_LLM_CACHE_DIR}')

# ── HuggingFace token ────────────────────────────────────────────────────────
# Read from the environment first; fall back to Colab Secrets. Never hardcode.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN not found.\n'
        '1) Accept the model licence on its HuggingFace page\n'
        '2) Create a Read token at huggingface.co/settings/tokens\n'
        '3) export HF_TOKEN=hf_...   (or add it to Colab Secrets, notebook access ON)'
    )
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN     # vLLM reads this name
print(f'HF_TOKEN loaded ({len(HF_TOKEN)} chars)')

# ── NVFP4 plugin patch (Gemma 4 NVFP4 only) ─────────────────────────────────
# The community Gemma-4 NVFP4 build needs a replacement for vLLM's gemma4.py.
# The exact file used for the paper's runs is vendored at vendor/gemma4_patched.py
# (sha256 recorded in README). We prefer the vendored copy; the upstream URL is
if _CFG.get('needs_nvfp4_patch'):
    import vllm, requests, hashlib
    vllm_dir   = os.path.dirname(vllm.__file__)
    patch_path = os.path.join(vllm_dir, 'model_executor', 'models', 'gemma4.py')
    vendored   = None
    for cand in ('vendor/gemma4_patched.py', '../vendor/gemma4_patched.py',
                 os.path.join(SDG_ROOT, 'vendor', 'gemma4_patched.py')):
        if os.path.exists(cand):
            vendored = cand
            break
    if vendored:
        text = open(vendored).read()
        print(f'NVFP4 patch: using vendored copy {vendored}')
    else:
        PATCH_URL = ('https://huggingface.co/bg-digitalservices/'
                     'Gemma-4-26B-A4B-it-NVFP4/resolve/main/gemma4_patched.py')
        print('NVFP4 patch: vendored copy not found, fetching upstream')
        r = requests.get(PATCH_URL, timeout=30)
        if r.status_code != 200 or len(r.text) < 1000:
            raise RuntimeError(
                f'NVFP4 patch unavailable (HTTP {r.status_code}). Place the file at '
                f'vendor/gemma4_patched.py, or set ACTIVE_MODEL to a backbone that '
                f'does not need it (mistral-24b / qwen3-32b / mixtral-8x7b).')
        text = r.text
    with open(patch_path, 'w') as f:
        f.write(text)
    print(f'NVFP4 patch applied to {patch_path}')
    print(f'  sha256: {hashlib.sha256(text.encode()).hexdigest()}')
else:
    print('NVFP4 patch not required for this backbone.')

VLLM_LOG = os.path.join(RESULTS_DIR, 'vllm_server.log')


---
## 1. Load Aurora dataset with target-level labels

In [ ]:
# Aurora dataset provides BOTH goal-level (17 columns) and target-level (169
# columns) labels on Zenodo 5224005. The v5 pipeline only loaded goal-level;
# here we load both so we can evaluate at both resolutions.

AURORA_GOALS_CACHE   = os.path.join(DATA_CACHE, 'aurora_sdg_goals.parquet')
AURORA_TARGETS_CACHE = os.path.join(DATA_CACHE, 'aurora_sdg_targets.parquet')
AURORA_SAMPLE_CACHE  = os.path.join(DATA_CACHE, 'aurora_multilabel_full.parquet')

def load_aurora_both_levels():
    """Fetch Aurora dataset; derive goal labels from target labels.

    Aurora v5 column conventions:
      - Goal columns:   'SDG01' .. 'SDG17'  (zero-padded, prefix 'SDG')
      - Target columns: '1.1' .. '17.19'    (bare target IDs)
      - Values: 1.0 if matched, NaN otherwise
    """
    if os.path.exists(AURORA_SAMPLE_CACHE):
        df = pd.read_parquet(AURORA_SAMPLE_CACHE)
        n_goal = sum(c.startswith('sdg_') for c in df.columns)
        n_tgt  = sum(c.startswith('target_') for c in df.columns)
        print(f'Loaded cached sample: {len(df)} papers, {n_goal} goal cols, {n_tgt} target cols')
        return df

    print('Downloading Aurora SDG target-level dataset from Zenodo...')
    os.makedirs('/tmp/aurora_dl', exist_ok=True)
    os.system('zenodo_get 10.5281/zenodo.5224005 -o /tmp/aurora_dl/ 2>&1 | tail -8')

    import glob
    files = sorted(glob.glob('/tmp/aurora_dl/*.csv'))
    print(f'Downloaded: {[os.path.basename(f) for f in files]}')

    # Pick the wide-format full target file (1.1M rows)
    wide_files = [f for f in files if 'in-columns' in f]
    full_wide  = [f for f in wide_files if 'SAMPLE' not in os.path.basename(f)]
    chosen     = (full_wide or wide_files)[0]
    print(f'Using: {os.path.basename(chosen)}')

    df_t = pd.read_csv(chosen, low_memory=False)
    print(f'  Loaded {len(df_t)} rows × {len(df_t.columns)} columns')

    # Rename goals: 'SDG01' → 'sdg_1', etc.
    rename_g = {f'SDG{g:02d}': f'sdg_{g}' for g in range(1, 18)
                if f'SDG{g:02d}' in df_t.columns}
    df_t = df_t.rename(columns=rename_g)

    # Rename targets: '1.1' → 'target_1_1', '17.a' → 'target_17_a'
    rename_t = {}
    for c in df_t.columns:
        m = re.fullmatch(r'(\d{1,2})\.([\w\d]+)', str(c))
        if m:
            rename_t[c] = f'target_{m.group(1)}_{m.group(2)}'
    df_t = df_t.rename(columns=rename_t)

    goal_cols   = sorted([c for c in df_t.columns if re.fullmatch(r'sdg_\d+', c)],
                         key=lambda c: int(c.split('_')[1]))
    target_cols = sorted([c for c in df_t.columns if c.startswith('target_')])
    print(f'  Found {len(goal_cols)} goal columns, {len(target_cols)} target columns')

    if not goal_cols or not target_cols:
        raise RuntimeError(f'Column matching failed. Sample columns: {list(df_t.columns[:25])}')

    # NaN → 0, ≥1 → 1
    for c in goal_cols + target_cols:
        df_t[c] = df_t[c].fillna(0).astype(int).clip(0, 1)

    df_t['doi'] = df_t['doi'].astype(str).str.strip().str.lower()
    df_t = df_t.loc[:, ~df_t.columns.duplicated()]

    df_t['n_goals'] = df_t[goal_cols].sum(axis=1)
    df_multi = df_t[df_t['n_goals'] >= 2].reset_index(drop=True)
    print(f'Multi-label subset (≥2 goals): {len(df_multi)} papers (from {len(df_t)} total)')

    df_multi.to_parquet(AURORA_SAMPLE_CACHE, index=False)
    return df_multi

df_multi = load_aurora_both_levels()
goal_cols   = sorted([c for c in df_multi.columns if c.startswith('sdg_')],
                     key=lambda c: int(c.split('_')[-1]))
target_cols = sorted([c for c in df_multi.columns if c.startswith('target_')])
print(f'\nGoal columns:   {len(goal_cols)}')
print(f'Target columns: {len(target_cols)}')
print(f'Multi-label papers: {len(df_multi)}  avg goals/paper: {df_multi["n_goals"].mean():.2f}')


In [ ]:
if FORCE_REFRESH:
    p = os.path.join(DATA_CACHE, 'aurora_multilabel_full.parquet')
    if os.path.exists(p):
        os.remove(p); print(f'Removed {p}')
else:
    print('Skipping aurora_multilabel_full.parquet wipe (FORCE_REFRESH=False).')


---
## 2. Fetch abstracts via OpenAlex (same as v5)

In [ ]:
OPENALEX_MAILTO = os.environ.get('OPENALEX_MAILTO', 'sdg-research@example.com')
ABSTRACTS_CACHE = os.path.join(DATA_CACHE, 'abstracts_full.parquet')  # full: N_FEWSHOT+N_VAL+N_TEST docs

def fetch_abstracts_openalex(dois, batch_size=50):
    """Batch-fetch title+abstract from OpenAlex. Returns dict doi→{title,abstract}."""
    results = {}
    dois = [d.strip().lower() for d in dois if d and str(d) != 'nan']
    for i in range(0, len(dois), batch_size):
        batch = dois[i:i+batch_size]
        pipe_dois = '|'.join(batch)
        url = (f'https://api.openalex.org/works'
               f'?filter=doi:{pipe_dois}'
               f'&select=doi,title,abstract_inverted_index'
               f'&per-page={batch_size}&mailto={OPENALEX_MAILTO}')
        try:
            r = requests.get(url, timeout=30, headers={'User-Agent': 'sdg-research/1.0'})
            if r.status_code == 200:
                for w in r.json().get('results', []):
                    doi = w.get('doi','').replace('https://doi.org/','').lower()
                    title = w.get('title') or ''
                    # Reconstruct abstract from inverted index
                    inv = w.get('abstract_inverted_index') or {}
                    if inv:
                        idx2word = {}
                        for word, positions in inv.items():
                            for pos in positions:
                                idx2word[pos] = word
                        abstract = ' '.join(idx2word[k] for k in sorted(idx2word))
                    else:
                        abstract = ''
                    if title:
                        results[doi] = {'title': title, 'abstract': abstract}
        except Exception as e:
            print(f'  Batch {i//batch_size} error: {e}')
        time.sleep(0.1)  # polite rate limiting
        if (i // batch_size) % 5 == 0:
            print(f'  Fetched {min(i+batch_size, len(dois))}/{len(dois)} abstracts...')
    return results

if os.path.exists(ABSTRACTS_CACHE):
    df_papers = pd.read_parquet(ABSTRACTS_CACHE)
    print(f'Loaded cached papers with abstracts: {len(df_papers)}')
else:
    # Sample: stratified by number of goals (prefer 2-3 goal papers, not outliers)
    n_total = N_FEWSHOT + N_VAL + N_TEST
    df_strat = df_multi[df_multi['n_goals'].between(2, 5)]
    if len(df_strat) < n_total:
        df_strat = df_multi
    # Oversample 2x only — 5x caused 3250 OpenAlex fetches and slow ETA
    df_sampled = df_strat.sample(min(n_total * 2, len(df_strat)),
                                  random_state=SEED)

    print(f'Fetching abstracts for {len(df_sampled)} candidate papers from OpenAlex...')
    abstract_dict = fetch_abstracts_openalex(df_sampled['doi'].tolist())
    print(f'Got abstracts for {len(abstract_dict)}/{len(df_sampled)} papers')

    # Join abstracts
    df_sampled['doi_lower'] = df_sampled['doi'].str.strip().str.lower()
    df_sampled['title']    = df_sampled['doi_lower'].map(lambda d: abstract_dict.get(d, {}).get('title',''))
    df_sampled['abstract'] = df_sampled['doi_lower'].map(lambda d: abstract_dict.get(d, {}).get('abstract',''))
    df_sampled = df_sampled[df_sampled['title'].str.len() > 10].reset_index(drop=True)

    # Take final sample
    df_papers = df_sampled.head(n_total).reset_index(drop=True)
    df_papers.to_parquet(ABSTRACTS_CACHE, index=False)
    print(f'Saved {len(df_papers)} papers with abstracts (fewshot+val+test).')

goal_cols = [c for c in df_papers.columns if c.startswith('sdg_')]
df_fewshot = df_papers.iloc[:N_FEWSHOT].reset_index(drop=True)
df_val     = df_papers.iloc[N_FEWSHOT:N_FEWSHOT+N_VAL].reset_index(drop=True)
df_test    = df_papers.iloc[N_FEWSHOT+N_VAL:N_FEWSHOT+N_VAL+N_TEST].reset_index(drop=True)
val_labels  = df_val[goal_cols].values.astype(int)
test_labels = df_test[goal_cols].values.astype(int)

# Hard checks — catch silent size errors before spending API credits
assert len(df_fewshot) == N_FEWSHOT, f'Expected {N_FEWSHOT} fewshot, got {len(df_fewshot)}'
assert len(df_val)     == N_VAL,     f'Expected {N_VAL} val, got {len(df_val)}'
assert len(df_test)    == N_TEST,     f'Expected {N_TEST} test, got {len(df_test)}'
print(f'Few-shot: {len(df_fewshot)}  Val (calibration): {len(df_val)}  Test: {len(df_test)}')
print(f'Val  multi-label fraction: {(val_labels.sum(1)>1).mean()*100:.1f}%')
print(f'Test multi-label fraction: {(test_labels.sum(1)>1).mean()*100:.1f}%')
print(f'Avg goals/test doc: {test_labels.sum(1).mean():.2f}')

import hashlib
def _doi_fingerprint(df, n=5):
    dois = df['doi'].astype(str).tolist()
    head = dois[:n]
    full = ''.join(dois).encode('utf-8')
    h = hashlib.sha1(full).hexdigest()[:12]
    return head, h
_head_test, _hash_test = _doi_fingerprint(df_test)
_head_val,  _hash_val  = _doi_fingerprint(df_val)
print(f'\n── Data fingerprint (verify across model runs) ──')
print(f'  test split: hash={_hash_test}  first 5 DOIs={_head_test}')
print(f'  val  split: hash={_hash_val}   first 5 DOIs={_head_val}')


---
## 3. SDG target taxonomy (169 targets) with short descriptors

In [ ]:
# Official UN 2030 Agenda target IDs. 169 targets total distributed across 17 goals.
# Means-of-implementation targets use letters (1.a, 1.b, ..., 17.a-f). Short descriptors
# are abbreviated for prompt efficiency — full wording is at https://sdgs.un.org/goals.

TARGET_LIST = [
    ('1.1','eradicate extreme poverty'), ('1.2','reduce poverty by half'), ('1.3','social protection systems'),
    ('1.4','equal economic rights & basic services'), ('1.5','resilience to climate/economic shocks'),
    ('1.a','mobilise resources for poverty programs'), ('1.b','pro-poor policy frameworks'),
    ('2.1','end hunger, universal food access'), ('2.2','end malnutrition'), ('2.3','double smallholder productivity'),
    ('2.4','sustainable food production systems'), ('2.5','genetic diversity of seeds/livestock'),
    ('2.a','investment in agricultural infrastructure'), ('2.b','correct trade distortions in ag'),
    ('2.c','food commodity market stability'),
    ('3.1','reduce maternal mortality'), ('3.2','end preventable child/newborn death'), ('3.3','end AIDS/TB/malaria/NTDs'),
    ('3.4','reduce non-communicable disease'), ('3.5','substance abuse prevention'), ('3.6','halve road traffic deaths'),
    ('3.7','universal reproductive health access'), ('3.8','universal health coverage'), ('3.9','reduce pollution-related illness'),
    ('3.a','tobacco control'), ('3.b','vaccine/medicine R&D access'),
    ('3.c','health workforce in developing countries'), ('3.d','global health risk management'),
    ('4.1','free primary + secondary education'), ('4.2','early childhood development'), ('4.3','equal access to TVET + tertiary'),
    ('4.4','skills for employment'), ('4.5','gender/disability parity in education'), ('4.6','adult literacy + numeracy'),
    ('4.7','education for sustainable development'), ('4.a','inclusive school facilities'),
    ('4.b','scholarships for developing countries'), ('4.c','qualified teacher supply'),
    ('5.1','end discrimination against women'), ('5.2','eliminate violence against women'), ('5.3','end child marriage/FGM'),
    ('5.4','unpaid care work recognition'), ('5.5','women in leadership'), ('5.6','sexual/reproductive rights'),
    ('5.a','equal rights to economic resources'), ('5.b','women + ICT'), ('5.c','gender equality policy frameworks'),
    ('6.1','safe drinking water for all'), ('6.2','sanitation + hygiene'), ('6.3','reduce water pollution'),
    ('6.4','water-use efficiency'), ('6.5','integrated water resources management'),
    ('6.6','protect water-related ecosystems'), ('6.a','water + sanitation international cooperation'),
    ('6.b','community participation in water management'),
    ('7.1','universal modern energy access'), ('7.2','increase renewable energy share'),
    ('7.3','double energy efficiency rate'), ('7.a','clean energy research access'),
    ('7.b','energy infrastructure in developing countries'),
    ('8.1','sustain GDP growth'), ('8.2','economic diversification + innovation'), ('8.3','decent job creation'),
    ('8.4','resource efficiency in consumption/production'), ('8.5','full + productive employment'),
    ('8.6','reduce youth unemployment'), ('8.7','end forced labour + child labour'), ('8.8','protect labour rights'),
    ('8.9','sustainable tourism'), ('8.10','financial services access'),
    ('8.a','aid-for-trade'), ('8.b','youth employment global strategy'),
    ('9.1','resilient infrastructure'), ('9.2','inclusive sustainable industrialisation'),
    ('9.3','small enterprise financial services'), ('9.4','upgrade for sustainability'),
    ('9.5','enhance R&D'), ('9.a','sustainable infrastructure in developing countries'),
    ('9.b','domestic tech development'), ('9.c','universal ICT access'),
    ('10.1','income growth for bottom 40%'), ('10.2','social/economic inclusion'), ('10.3','equal opportunity'),
    ('10.4','fiscal + wage policy for equality'), ('10.5','regulate financial markets'),
    ('10.6','developing country voice in institutions'), ('10.7','safe migration'),
    ('10.a','special treatment for developing countries'), ('10.b','development assistance to where needed'),
    ('10.c','reduce remittance transaction costs'),
    ('11.1','safe affordable housing'), ('11.2','sustainable transport'), ('11.3','inclusive urbanisation'),
    ('11.4','safeguard cultural/natural heritage'), ('11.5','reduce disaster casualties/damage'),
    ('11.6','reduce urban environmental impact'), ('11.7','accessible green spaces'),
    ('11.a','urban-rural links'), ('11.b','integrated disaster risk policies'),
    ('11.c','least-developed countries building support'),
    ('12.1','sustainable consumption/production framework'), ('12.2','sustainable natural resource management'),
    ('12.3','halve food waste'), ('12.4','sound chemical/waste management'), ('12.5','reduce waste via recycling'),
    ('12.6','corporate sustainability reporting'), ('12.7','sustainable public procurement'),
    ('12.8','sustainability awareness'),
    ('12.a','scientific/tech capacity in developing countries'), ('12.b','sustainable tourism monitoring'),
    ('12.c','rationalise fossil fuel subsidies'),
    ('13.1','strengthen climate resilience'), ('13.2','integrate climate in policy'), ('13.3','climate education + awareness'),
    ('13.a','Green Climate Fund commitments'), ('13.b','climate planning in LDCs'),
    ('14.1','reduce marine pollution'), ('14.2','protect coastal ecosystems'), ('14.3','reduce ocean acidification'),
    ('14.4','end overfishing'), ('14.5','conserve 10% coastal/marine areas'), ('14.6','eliminate harmful fisheries subsidies'),
    ('14.7','economic benefits for SIDS/LDCs from marine resources'),
    ('14.a','marine science + research capacity'), ('14.b','small-scale fishers market access'),
    ('14.c','implement UNCLOS'),
    ('15.1','conserve terrestrial ecosystems'), ('15.2','sustainable forest management'), ('15.3','combat desertification'),
    ('15.4','conserve mountain ecosystems'), ('15.5','halt biodiversity loss'), ('15.6','genetic resources access + benefits'),
    ('15.7','end poaching + trafficking of species'), ('15.8','control invasive species'),
    ('15.9','integrate ecosystem values in planning'),
    ('15.a','financial resources for biodiversity'), ('15.b','forest management financing'),
    ('15.c','combat wildlife poaching globally'),
    ('16.1','reduce violence + death rates'), ('16.2','end abuse/exploitation of children'), ('16.3','rule of law + justice access'),
    ('16.4','reduce illicit financial flows'), ('16.5','reduce corruption'), ('16.6','effective accountable institutions'),
    ('16.7','responsive/inclusive decision-making'), ('16.8','developing country participation in governance'),
    ('16.9','legal identity for all'), ('16.10','public access to information'),
    ('16.a','prevent violence + combat terrorism'), ('16.b','non-discriminatory laws'),
    ('17.1','domestic resource mobilisation'), ('17.2','ODA commitments'), ('17.3','additional financial resources for developing'),
    ('17.4','debt sustainability for LDCs'), ('17.5','investment promotion for LDCs'),
    ('17.6','North-South/South-South technology cooperation'), ('17.7','environmentally sound tech transfer'),
    ('17.8','tech bank for LDCs'), ('17.9','capacity-building support'),
    ('17.10','multilateral trading system'), ('17.11','increase developing country exports'),
    ('17.12','duty-free market access for LDCs'),
    ('17.13','macroeconomic stability'), ('17.14','policy coherence for sustainable development'),
    ('17.15','respect national policy space'),
    ('17.16','global partnership for sustainable development'), ('17.17','public-private + civil society partnerships'),
    ('17.18','high-quality disaggregated data'), ('17.19','measurements beyond GDP'),
]
assert len(TARGET_LIST) == 169, f'Expected 169 targets, got {len(TARGET_LIST)}'

TARGET_IDS  = [t[0] for t in TARGET_LIST]
TARGET_DESC = {t[0]: t[1] for t in TARGET_LIST}
# Hierarchy: target "1.2" belongs to goal 1, "17.a" belongs to goal 17
def target_to_goal(tid):
    return int(tid.split('.')[0])
TARGET_GOAL = np.array([target_to_goal(t) for t in TARGET_IDS])  # 1..17 for each target
GOAL_TO_TARGETS = {g: [i for i, t in enumerate(TARGET_IDS) if target_to_goal(t) == g]
                   for g in range(1, 18)}
print(f'169 targets distributed across 17 goals:')
for g in range(1, 18):
    print(f'  SDG {g:2d} ({SDG_NAMES[g]:25s}): {len(GOAL_TO_TARGETS[g])} targets')


In [ ]:
# Align Aurora's target columns (which may have various naming schemes) with
# our canonical 169-target ordering. This gives us test_target_labels of shape (N, 169).

def align_target_columns(df, canonical_target_ids):
    '''Build a (N, 169) label matrix from whatever target columns df actually has.'''
    # Build a lookup from various naming schemes to canonical IDs
    aliases = {}
    for tid in canonical_target_ids:
        g, sub = tid.split('.')
        # Possible column names in Aurora
        for cand in [f'target_{g}_{sub}', f'sdg_{g}_{sub}',
                     f'target_{g}.{sub}', f'sdg_{g}.{sub}',
                     f'target_{g.zfill(2)}_{sub}', f'sdg_{g.zfill(2)}_{sub}',
                     f'target_{tid}', f'sdg_{tid}', tid]:
            aliases[cand.lower()] = tid

    df_cols_lower = {c.lower(): c for c in df.columns}
    n_matched = 0
    label_matrix = np.zeros((len(df), 169), dtype=int)
    missing = []
    for j, tid in enumerate(canonical_target_ids):
        # Try each alias form
        matched_col = None
        for alias in aliases:
            if aliases[alias] == tid and alias in df_cols_lower:
                matched_col = df_cols_lower[alias]
                break
        if matched_col is None:
            missing.append(tid)
            continue
        label_matrix[:, j] = df[matched_col].astype(int).values
        n_matched += 1
    print(f'Target column alignment: {n_matched}/169 matched')
    if missing:
        print(f'  Missing targets ({len(missing)}): {missing[:10]}{"..." if len(missing) > 10 else ""}')
    return label_matrix

# Sample: stratified by n_goals (prefer 2-4 goal papers)
n_total = N_FEWSHOT + N_VAL + N_TEST
df_strat = df_multi[df_multi['n_goals'].between(2, 5)]
if len(df_strat) < n_total:
    df_strat = df_multi
df_papers_base = df_strat.sample(min(n_total * 2, len(df_strat)), random_state=SEED)
print(f'Sampled {len(df_papers_base)} candidate papers (oversample 2x for OpenAlex coverage)')

# Now join abstracts (reuse cached abstracts_full.parquet from previous cell)
if 'df_papers' not in dir():
    df_papers = pd.read_parquet(os.path.join(DATA_CACHE, 'abstracts_full.parquet'))
    print(f'Loaded abstracts_full.parquet: {len(df_papers)} papers with title+abstract')

# Ensure target columns are retained on df_papers (they should be from the merge)
has_target_cols = any(c.startswith('target_') or re.match(r'sdg_\d+_', c) for c in df_papers.columns)
if not has_target_cols:
    # Re-merge target-level labels onto df_papers by doi
    print('Re-merging target labels onto abstracts-enriched sample...')
    target_only_cols = [c for c in df_multi.columns if c.startswith('target_') or re.match(r'sdg_\d+_', c)]
    df_papers = df_papers.merge(
        df_multi[['doi'] + target_only_cols],
        on='doi', how='left'
    )

df_fewshot = df_papers.iloc[:N_FEWSHOT].reset_index(drop=True)
df_val     = df_papers.iloc[N_FEWSHOT:N_FEWSHOT+N_VAL].reset_index(drop=True)
df_test    = df_papers.iloc[N_FEWSHOT+N_VAL:N_FEWSHOT+N_VAL+N_TEST].reset_index(drop=True)

val_labels_goal  = df_val[goal_cols].values.astype(int)
test_labels_goal = df_test[goal_cols].values.astype(int)
val_labels_target  = align_target_columns(df_val,  TARGET_IDS)
test_labels_target = align_target_columns(df_test, TARGET_IDS)

print(f'\nSplit sizes: fewshot={len(df_fewshot)} val={len(df_val)} test={len(df_test)}')
print(f'Goal-level labels:   val {val_labels_goal.shape}  test {test_labels_goal.shape}')
print(f'Target-level labels: val {val_labels_target.shape}  test {test_labels_target.shape}')
print(f'Target label density: {test_labels_target.mean()*100:.2f}% of cells positive')


In [ ]:
import json, numpy as np, os

# LOCAL_MATRIX is set in Cell 5 to GRAPH_DIR/sdg_interaction_matrix_<MATRIX_VERSION>.json
print(f'Loading SDG interaction matrix from: {LOCAL_MATRIX}')

if not os.path.exists(LOCAL_MATRIX):
    raise FileNotFoundError(
        f'Expected the v3.2 Pradhan/Warchold matrix at:\n'
        f'  {LOCAL_MATRIX}\n\n'
        f'Run SDG_Matrix_Preprocessor_UN_v3_2.ipynb first to produce it.\n'
        f'The companion PNG (Figure 2 in the paper) is written next to the JSON\n'
        f'as sdg_interaction_matrix_{MATRIX_VERSION}.png.'
    )

with open(LOCAL_MATRIX) as f:
    sdg_matrix = json.load(f)
W_check = np.array(sdg_matrix.get('matrix_W') or sdg_matrix.get('matrix'))
assert W_check.shape == (17, 17), f'Expected 17x17 matrix, got {W_check.shape}'
print(f'Loaded sdg_matrix from {LOCAL_MATRIX}')
print(f'  top-level keys: {list(sdg_matrix.keys())}')
print(f'  matrix_W shape: {W_check.shape}')
print(f'  matrix_W range: [{W_check.min():+.3f}, {W_check.max():+.3f}]')
print(f'  off-diagonal nonzero entries: {int((W_check != 0).sum() - np.trace(W_check != 0))}')


---
## 4. Build the seven interaction matrices (A-G)

In [ ]:
# === Step 4: build all seven matrices (A-G) ===
# This cell does NOT call the LLM. The matrices are pure data:
#   A is loaded from your existing Pradhan output
#   B is derived from A by hierarchical inheritance
#   C is computed inside this cell from SBERT embeddings of target descriptions
#   D is constructed during grid search (Cell 19) as the best w-weighted
#     combination of B and C; its definition lives here for documentation.
#
# After this cell runs, all four matrices are in memory and the rest of the

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ── Matrix A: Pradhan goal-level (loaded in Cell 9 earlier) ──────────────────
W_pradhan_17 = np.array(sdg_matrix.get('matrix_W') or sdg_matrix.get('matrix'))
assert W_pradhan_17.shape == (17, 17), f'Expected Pradhan 17x17, got {W_pradhan_17.shape}'
print(f'Matrix A (Pradhan 17x17): range [{W_pradhan_17.min():.2f}, {W_pradhan_17.max():.2f}]')

# ── Matrix B: Pradhan inherited ──────────────────────────────────────────────
# For target pair (i, j) in different goals: W[i,j] = Pradhan[goal_i, goal_j]
# For target pair in same goal: W[i,j] = 0 (Pradhan carries no within-goal info)
W_inherit_169 = np.zeros((169, 169))
for i in range(169):
    for j in range(169):
        g_i, g_j = TARGET_GOAL[i], TARGET_GOAL[j]
        if g_i != g_j:
            W_inherit_169[i, j] = W_pradhan_17[g_i-1, g_j-1]
print(f'Matrix B (Pradhan inherited 169x169): range [{W_inherit_169.min():.2f}, {W_inherit_169.max():.2f}] '
      f'| {(W_inherit_169 != 0).mean()*100:.0f}% nonzero')

# ── Matrix C: Semantic similarity from SBERT embeddings ──────────────────────
# Methodology follows Song et al. (2023) Sustainable Development 31(4): 2784-2796,
# but uses sentence-transformers (Reimers & Gurevych 2019) instead of Word2Vec
# for stronger sentence-level semantics. The 169 target descriptors were
# defined in Cell 9 as TARGET_DESC. We compute pairwise cosine similarity,
# zero the diagonal, and centre the distribution so the matrix has both
# positive (similar) and negative (dissimilar) entries — matching A's and B's
# scale conventions for downstream signed propagation.

SBERT_CACHE = os.path.join(DATA_CACHE, 'sbert_target_matrix_169.npy')
if os.path.exists(SBERT_CACHE):
    W_semantic_169 = np.load(SBERT_CACHE)
    print(f'Loaded cached SBERT semantic matrix from {SBERT_CACHE}')
else:
    print('Computing SBERT embeddings for 169 SDG target descriptions...')
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer('all-MiniLM-L6-v2')

    # Use full descriptors (target ID + descriptor) so the model has more context
    # than the bare descriptor alone.
    texts = [f'SDG target {tid}: {TARGET_DESC[tid]}' for tid in TARGET_IDS]
    embeddings = sbert.encode(texts, show_progress_bar=False, normalize_embeddings=True)

    W_semantic_raw = cosine_similarity(embeddings)         # in [-1, 1] but typically [0.2, 1.0]
    np.fill_diagonal(W_semantic_raw, 0)

    # Centre by median of off-diagonal entries so we get both signs.
    # This is the Song-2023-equivalent rescaling: pairs more similar than
    # the typical pair become positive, pairs less similar become negative.
    off_diag = W_semantic_raw[~np.eye(169, dtype=bool)]
    centre = np.median(off_diag)
    W_semantic_169 = W_semantic_raw - centre
    np.fill_diagonal(W_semantic_169, 0)

    # Rescale to match Pradhan's typical magnitude range
    target_max = max(abs(W_inherit_169.min()), abs(W_inherit_169.max()))
    current_max = max(abs(W_semantic_169.min()), abs(W_semantic_169.max()))
    if current_max > 0 and target_max > 0:
        W_semantic_169 = W_semantic_169 * (target_max / current_max)

    np.save(SBERT_CACHE, W_semantic_169)
    print(f'Cached SBERT matrix to {SBERT_CACHE}')

print(f'Matrix C (semantic 169x169): range [{W_semantic_169.min():.2f}, {W_semantic_169.max():.2f}] '
      f'| {(W_semantic_169 > 0).mean()*100:.0f}% positive')

# ── Matrix D: combined B + C (defined here, weight calibrated in Cell 19) ───
# W_D = w_B * W_inherit_169 + (1 - w_B) * W_semantic_169
# w_B is searched over {0.0, 0.25, 0.5, 0.75, 1.0} during grid search;
# the single best w_B (selected on val F1) is reported as part of D's results.
def build_combined_matrix(w_B):
    """Return W_D = w_B * B + (1 - w_B) * C; both B and C are 169x169."""
    return w_B * W_inherit_169 + (1.0 - w_B) * W_semantic_169

# ── Register all four matrices for the comparison loop ───────────────────────
# Note: D is registered as a callable factory because its weight is searched
# at evaluation time. The eval cell (Cell 19) handles this specially.

# ── Matrix E: UN hierarchical/taxonomic (169x169) ────────────────────────────
# Ontology-style matrix from the UN SDG taxonomy: targets within the same goal
# are taxonomically closest; goals within the same UN "5 P's" cluster (People,
# Planet, Prosperity, Peace, Partnership) are medium-related; cross-cluster
# pairs are distant.
SDG_CLUSTERS = {
    'people':       {1, 2, 3, 4, 5},
    'planet':       {6, 12, 13, 14, 15},
    'prosperity':   {7, 8, 9, 10, 11},
    'peace':        {16},
    'partnership':  {17},
}
GOAL_TO_CLUSTER = {}
for cluster_name, goals in SDG_CLUSTERS.items():
    for g in goals:
        GOAL_TO_CLUSTER[g] = cluster_name

W_SAMEGOAL  =  0.35
W_SAMEP     =  0.20
W_DIFFP     =  0.05

W_taxonomy_169 = np.zeros((169, 169))
for i in range(169):
    gi = TARGET_GOAL[i]; ci = GOAL_TO_CLUSTER[gi]
    for j in range(169):
        if i == j: continue
        gj = TARGET_GOAL[j]; cj = GOAL_TO_CLUSTER[gj]
        if gi == gj:    W_taxonomy_169[i, j] = W_SAMEGOAL
        elif ci == cj:  W_taxonomy_169[i, j] = W_SAMEP
        else:           W_taxonomy_169[i, j] = W_DIFFP
print(f'Matrix E (UN taxonomy 169x169): range '
      f'[{W_taxonomy_169.min():.2f}, {W_taxonomy_169.max():.2f}]')

# ── Matrix F: Keyword-overlap (169x169) ──────────────────────────────────────
import re as _re
_STOPWORDS_TEXT = (
    'a an and are as at be by for from has have he her him his '
    'i in is it its of on or our she that the their them they this to was we were '
    'will with you your via per through into onto upon during such other than then '
    'which when where what who whom whose why how all some any most much many few '
    'also more less least best own no not nor only own so too very same just both '
    'each every either neither each been being do does did doing have having had'
)
STOPWORDS = set(_STOPWORDS_TEXT.split())

def _extract_keywords(text):
    text = _re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    return set(t for t in text.split() if len(t) > 2 and t not in STOPWORDS)

target_keywords = {tid: _extract_keywords(TARGET_DESC[tid]) for tid in TARGET_IDS}

W_keyword_169 = np.zeros((169, 169))
for i, ti in enumerate(TARGET_IDS):
    for j, tj in enumerate(TARGET_IDS):
        if i == j: continue
        union = target_keywords[ti] | target_keywords[tj]
        if not union: continue
        W_keyword_169[i, j] = len(target_keywords[ti] & target_keywords[tj]) / len(union)

# Centre and rescale to Pradhan range
off = W_keyword_169[~np.eye(169, dtype=bool)]
W_keyword_169 = W_keyword_169 - np.median(off)
np.fill_diagonal(W_keyword_169, 0)
target_max = max(abs(W_inherit_169.min()), abs(W_inherit_169.max()))
current_max = max(abs(W_keyword_169.min()), abs(W_keyword_169.max()))
if current_max > 0 and target_max > 0:
    W_keyword_169 = W_keyword_169 * (target_max / current_max)
print(f'Matrix F (keyword-overlap 169x169): range '
      f'[{W_keyword_169.min():.2f}, {W_keyword_169.max():.2f}]')

# ── Matrix G: random Gaussian baseline ───────────────────────────────────────
# A scale-matched Gaussian noise matrix serving as the methodological floor.
# If real graphs A-F don't beat this, the propagation step itself isn't doing
# meaningful work — useful sanity check for any reviewer.
rng = np.random.RandomState(SEED)
W_random_169 = rng.normal(loc=0.0, scale=W_inherit_169.std(), size=(169, 169))
W_random_169 = (W_random_169 + W_random_169.T) / 2     # symmetrise
np.fill_diagonal(W_random_169, 0)
print(f'Matrix G (random Gaussian baseline 169x169): range '
      f'[{W_random_169.min():.2f}, {W_random_169.max():.2f}]')

GRAPHS = {
    'A_pradhan_17':    ('Pradhan 17x17 (goal-level baseline)',           W_pradhan_17,  17),
    'B_inherit_169':   ('Pradhan inherited 169x169',                     W_inherit_169, 169),
    'C_semantic_169':  ('Semantic similarity 169x169 (SBERT)',           W_semantic_169, 169),
    'D_combined_169':  ('Combined 169x169 (w·B + (1-w)·C, w on val)',   None,          169),
    'E_taxonomy_169':  ('UN taxonomic 169x169 (5P clusters)',            W_taxonomy_169, 169),
    'F_keyword_169':   ('Keyword overlap 169x169 (lexical-symbolic)',    W_keyword_169, 169),
    'G_random_169':    ('Random Gaussian baseline 169x169 (sanity floor)', W_random_169, 169),
}

print(f'\n{len(GRAPHS)} graphs registered for comparison:')
for k, (name, W, dim) in GRAPHS.items():
    if W is None:
        print(f'  {k:20s}: {name} ({dim}x{dim})  [calibrated at eval time]')
    else:
        print(f'  {k:20s}: {name} ({dim}x{dim})')


In [ ]:
print(f'(v5: legacy results_local_v3 cleanup skipped — RESULTS_DIR={RESULTS_DIR})')


---
## 5. Stage-1 LLM — score 17 goals per paper (concurrent)

In [ ]:
def make_fewshot_examples(df_fs, n=6):
    """Sample n few-shot examples from df_fs, format them for the prompt."""
    samples = df_fs.sample(min(n, len(df_fs)), random_state=SEED)
    blocks = []
    for _, row in samples.iterrows():
        title = str(row.get('title', ''))[:200]
        abstract = str(row.get('abstract', ''))[:600]
        labels = [int(row[c]) for c in goal_cols]
        active = [i+1 for i, v in enumerate(labels) if v]
        scores = {str(i+1): (0.9 if labels[i] else 0.05) for i in range(17)}
        blocks.append(
            f'TITLE: {title}\nABSTRACT: {abstract}\n'
            f'AURORA-LABELLED SDGS: {active}\n'
            f'OUTPUT: {{"scores": {json.dumps(scores)}, '
            f'"reasoning": "Maps to SDGs {active}."}}'
        )
    return '\n\n---\n\n'.join(blocks)

def build_prompt(papers_batch, fewshot_str):
    """Build a single prompt scoring multiple papers in one call."""
    sdg_list = '\n'.join(f'  {k}: {v}' for k, v in SDG_NAMES.items())
    papers = []
    for i, (_, row) in enumerate(papers_batch.iterrows()):
        title = str(row.get('title', ''))[:200]
        abstract = str(row.get('abstract', ''))[:600]
        papers.append(f'PAPER {i+1}:\nTITLE: {title}\nABSTRACT: {abstract}')
    return (
        f'You are classifying research papers by relevance to the 17 UN '
        f'Sustainable Development Goals.\n\nSDGs:\n{sdg_list}\n\n'
        f'For each paper below, output a JSON object on its own line with '
        f'this exact format:\n'
        f'{{"paper": N, "scores": {{"1": 0.0, ..., "17": 0.0}}, '
        f'"reasoning": "<one short sentence>"}}\n\n'
        f'Score each SDG 0.0 (irrelevant) to 1.0 (clearly central). Output the '
        f'scores BEFORE the reasoning so the JSON parses even if reasoning is '
        f'truncated.\n\nExamples:\n\n{fewshot_str}\n\n---\n\n'
        f'Now classify these papers:\n\n' + '\n\n'.join(papers) + '\n\n'
        f'Output exactly {len(papers_batch)} JSON object(s), one per line, no other text.'
    )

def build_prompt_single(row, fewshot_str):
    """Single-paper prompt for retry path."""
    sdg_list = '\n'.join(f'  {k}: {v}' for k, v in SDG_NAMES.items())
    title = str(row.get('title', ''))[:200]
    abstract = str(row.get('abstract', ''))[:600]
    return (
        f'Classify this paper by relevance to the 17 UN SDGs.\n\n'
        f'SDGs:\n{sdg_list}\n\nOutput a single JSON object:\n'
        f'{{"scores": {{"1": 0.0, ..., "17": 0.0}}, "reasoning": "<one short sentence>"}}\n\n'
        f'Examples:\n\n{fewshot_str}\n\n---\n\n'
        f'PAPER:\nTITLE: {title}\nABSTRACT: {abstract}\n\nOutput the JSON only.'
    )

def parse_batch_response(text, expected_n):
    """Extract list of (scores_dict, reasoning) tuples from LLM output."""
    results = []
    for line in text.split('\n'):
        line = line.strip()
        if not line.startswith('{'): continue
        m = re.search(r'\{.*\}', line)
        if not m: continue
        try:
            obj = json.loads(m.group(0))
            scores = obj.get('scores', {})
            reasoning = str(obj.get('reasoning', ''))[:300]
            clean_scores = {}
            for k, v in scores.items():
                try: clean_scores[str(k)] = float(v)
                except (ValueError, TypeError): clean_scores[str(k)] = 0.0
            results.append((clean_scores, reasoning))
        except Exception:
            continue
    # Pad with parse_error placeholders if too few were parsed
    while len(results) < expected_n:
        results.append(({str(k): 0.0 for k in range(1, 18)}, 'parse_error'))
    return results[:expected_n]

def parse_single_response(text):
    """Extract a single (scores_dict, reasoning) tuple."""
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m: return ({str(k): 0.0 for k in range(1, 18)}, 'parse_error')
    try:
        obj = json.loads(m.group(0))
        scores = obj.get('scores', {})
        reasoning = str(obj.get('reasoning', ''))[:300]
        clean_scores = {}
        for k, v in scores.items():
            try: clean_scores[str(k)] = float(v)
            except (ValueError, TypeError): clean_scores[str(k)] = 0.0
        return clean_scores, reasoning
    except Exception:
        return ({str(k): 0.0 for k in range(1, 18)}, 'parse_error')

def is_contaminated(prob_vec):
    if prob_vec.sum() < 0.01: return True
    if prob_vec.std() < 0.01: return True
    return False

def llm_call(prompt, max_tokens=2000):
    """Single OpenAI-compatible call. Honours per-model chat_template_kwargs
    (e.g. enable_thinking) and strips <think>...</think> blocks for R1-distills."""
    extra_body = {}
    ctk = _CFG.get('chat_template_kwargs')
    if ctk:
        if THINKING_MODE and 'enable_thinking' in ctk:
            ctk = {**ctk, 'enable_thinking': True}
        extra_body['chat_template_kwargs'] = ctk
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=max_tokens,
            temperature=0.0,
            extra_body=extra_body or None,
        )
        text = resp.choices[0].message.content or ''
        if _CFG.get('strip_think_blocks'):
            text = _strip_think(text)
        return text
    except Exception as e:
        return f'__ERROR__: {e}'

def call_llm_batch(papers_batch, fewshot_str, max_retries=3):
    """Score a batch of papers with one LLM call. Per-paper retry on partial parse.
    Returns list of (scores_dict, reasoning) tuples in input order."""
    prompt = build_prompt(papers_batch, fewshot_str)
    for attempt in range(max_retries):
        raw = llm_call(prompt, max_tokens=2000)
        if raw.startswith('__ERROR__'):
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            # Final attempt: per-paper retry
            results = []
            for _, row in papers_batch.iterrows():
                single = llm_call(build_prompt_single(row, fewshot_str), max_tokens=400)
                results.append(parse_single_response(single))
            return results
        return parse_batch_response(raw, len(papers_batch))


In [ ]:
# ── Stage-1 execution: score 17 goals for every paper (concurrent + checkpointed) ──

GOAL_CACHE_TEST = os.path.join(RESULTS_DIR, f'goal_probs_test_{VERSION}.npy')
GOAL_CACHE_VAL  = os.path.join(RESULTS_DIR, f'goal_probs_val_{VERSION}.npy')
GOAL_CHECKPOINT_TEST = os.path.join(RESULTS_DIR, f'goal_checkpoint_test_{VERSION}.jsonl')
GOAL_CHECKPOINT_VAL  = os.path.join(RESULTS_DIR, f'goal_checkpoint_val_{VERSION}.jsonl')

def load_checkpoint(path):
    if not os.path.exists(path): return {}
    done = {}
    with open(path) as f:
        for line in f:
            try:
                e = json.loads(line)
                done[e['batch_start']] = e['results']
            except Exception: pass
    return done

def run_llm_concurrent(df, fewshot_str, batch_size, concurrency, checkpoint_path, label):
    '''Concurrent version of call_llm_batch with JSONL checkpointing (resumable).'''
    batches = [(i, df.iloc[i:i+batch_size]) for i in range(0, len(df), batch_size)]
    n_batches = len(batches)
    done = load_checkpoint(checkpoint_path)
    if done: print(f'  [{label}] resuming — {len(done)}/{n_batches} already done')
    out = [None] * n_batches
    for pos, (start, _) in enumerate(batches):
        if start in done:
            out[pos] = [(r['scores'], r['reasoning']) for r in done[start]]
    pending = [(idx, b) for idx, b in batches if idx not in done]
    if not pending:
        flat = []; [flat.extend(br) for br in out]; return flat[:len(df)]

    f_ckpt = open(checkpoint_path, 'a'); lock = Lock()
    t0 = time.time(); completed = [len(done)]
    def work(start, b):
        r = call_llm_batch(b, fewshot_str)
        with lock:
            f_ckpt.write(json.dumps({'batch_start': int(start),
                                     'results': [{'scores': dict(s), 'reasoning': rs} for s, rs in r]}) + '\n')
            f_ckpt.flush(); completed[0] += 1
            if completed[0] % max(1, n_batches//30) == 0 or completed[0] == n_batches:
                el = time.time() - t0
                rate = (completed[0] - len(done)) / el if el > 0 else 0
                eta = (n_batches - completed[0]) / rate if rate > 0 else 0
                print(f'  [{label}] {completed[0]}/{n_batches}  {el:.0f}s  '
                      f'{rate*batch_size:.1f} docs/s  ETA {eta:.0f}s')
        return start, r

    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [ex.submit(work, idx, b) for idx, b in pending]
        for fut in as_completed(futures):
            start, r = fut.result()
            out[start // batch_size] = r
    f_ckpt.close()
    flat = []; [flat.extend(br) for br in out]; return flat[:len(df)]

if os.path.exists(GOAL_CACHE_TEST) and os.path.exists(GOAL_CACHE_VAL):
    goal_probs_test = np.load(GOAL_CACHE_TEST)
    goal_probs_val  = np.load(GOAL_CACHE_VAL)
    print(f'Loaded cached goal probs — test {goal_probs_test.shape}  val {goal_probs_val.shape}')
else:
    fewshot_str = make_fewshot_examples(df_fewshot, n=6)
    t0 = time.time()
    print(f'Stage 1 (goal-level) on val ({len(df_val)} docs)...')
    val_r = run_llm_concurrent(df_val, fewshot_str, BATCH_SIZE, CONCURRENCY,
                               GOAL_CHECKPOINT_VAL, 'stage1-val')
    goal_probs_val = np.array([[float(s.get(str(k), 0.0)) for k in range(1,18)]
                               for s, _ in val_r])
    np.save(GOAL_CACHE_VAL, goal_probs_val)

    print(f'\nStage 1 (goal-level) on test ({len(df_test)} docs)...')
    test_r = run_llm_concurrent(df_test, fewshot_str, BATCH_SIZE, CONCURRENCY,
                                GOAL_CHECKPOINT_TEST, 'stage1-test')
    goal_probs_test = np.array([[float(s.get(str(k), 0.0)) for k in range(1,18)]
                                for s, _ in test_r])
    np.save(GOAL_CACHE_TEST, goal_probs_test)
    # Also write to shared cache so _kg and _doc notebooks can reuse
    np.save(os.path.join(SHARED_LLM_CACHE_DIR, 'goal_probs_test.npy'), goal_probs_test)
    np.save(os.path.join(SHARED_LLM_CACHE_DIR, 'goal_probs_val.npy'), goal_probs_val)
    print(f'  Also saved to {SHARED_LLM_CACHE_DIR}/goal_probs_*.npy for downstream notebooks')
    print(f'\nStage 1 done in {time.time()-t0:.0f}s. '
          f'Val mean={goal_probs_val.mean():.3f}  Test mean={goal_probs_test.mean():.3f}')


---
## 6. Stage-2 LLM — score targets of confident goals

For each (paper, goal) pair where the stage-1 goal probability ≥ `STAGE2_THRESHOLD`, we ask the LLM to score each target under that goal (typically 5-15 targets per call). For goals below threshold, we initialise target probabilities by spreading the goal probability uniformly across its targets. This keeps cost bounded while still giving the graph a fine-grained signal where the LLM has something to say.


In [ ]:
# ── Stage-2 prompt + caller (BATCHED across confident goals per paper) ──

def build_target_prompt_split(row, confident_goals, target_lookup):
    # Build (cacheable_prefix, dynamic_suffix) for batched target scoring.
    # confident_goals: list of goal numbers (1-17) above STAGE2_THRESHOLD.
    # target_lookup: dict {goal: [target_id, ...]} listing the canonical
    # targets under each goal.

    # Static prefix — same wording every call, suitable for caching
    prefix = (
        'You are scoring research papers against UN SDG targets at the '
        'fine-grained 169-target level.\n\n'
        'For each block of targets below (one block per relevant SDG goal), '
        'assign each target a score 0.0 (irrelevant) to 1.0 (clearly central) '
        'based on the paper.\n\n'
        'Output ONLY a single JSON object with this exact format:\n'
        '{"target_scores": {"<goal>.<target>": <float 0-1>, ...}}\n\n'
        'Include EVERY target listed in the request. Use 0.0 for non-applicable.\n'
        'Do not add commentary outside the JSON.\n\n---\n\n'
    )

    # Dynamic suffix — paper-specific, NOT cached
    title    = str(row.get('title',''))[:200]
    abstract = str(row.get('abstract',''))[:600]
    target_blocks = []
    for g in sorted(confident_goals):
        tids = target_lookup.get(g, [])
        if not tids: continue
        lines = [f'  {tid}: {TARGET_DESC[tid]}' for tid in tids]
        target_blocks.append(
            f'### SDG {g}: {SDG_NAMES[g]}\n' + '\n'.join(lines)
        )
    suffix = (
        f'PAPER:\nTitle: {title}\nAbstract: {abstract}\n\n'
        f'TARGETS TO SCORE:\n\n' + '\n\n'.join(target_blocks) + '\n\n'
        f'Return the JSON now.'
    )
    return prefix, suffix

def call_llm_targets_batched(row, confident_goals, max_retries=3):
    """Score all targets under a paper's confident goals in ONE LLM call.
    Returns dict {target_id: prob}. Targets not in any confident goal stay 0.
    Uses vLLM's OpenAI-compatible client (no caching like Claude has — local LLM
    re-processes the prompt each call, but inference is free and fast)."""
    if not confident_goals:
        return {}
    target_lookup = {g: [TARGET_IDS[i] for i in GOAL_TO_TARGETS[g]]
                     for g in confident_goals}
    expected_tids = [t for tids in target_lookup.values() for t in tids]

    prefix, suffix = build_target_prompt_split(row, confident_goals, target_lookup)
    full_prompt = prefix + suffix

    extra_body = {}
    ctk = _CFG.get('chat_template_kwargs')
    if ctk:
        if THINKING_MODE and 'enable_thinking' in ctk:
            ctk = {**ctk, 'enable_thinking': True}
        extra_body['chat_template_kwargs'] = ctk
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{'role': 'user', 'content': full_prompt}],
                max_tokens=900,
                temperature=0.0,
                extra_body=extra_body or None,
            )
            raw = resp.choices[0].message.content or ''
            if _CFG.get('strip_think_blocks'):
                raw = _strip_think(raw)
            raw = raw.strip()
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if not m: raise ValueError('no JSON found')
            obj = json.loads(m.group(0))
            ts = obj.get('target_scores', {})
            return {tid: float(ts.get(tid, 0.0)) for tid in expected_tids}
        except Exception as e:
            if attempt == max_retries - 1:
                return {tid: 0.0 for tid in expected_tids}
            time.sleep(2 ** attempt)

# ── Stage 2 driver ──────────────────────────────────────────────────────────

def run_stage2(df_in, goal_probs, threshold=STAGE2_THRESHOLD,
               concurrency=CONCURRENCY, label='stage2',
               checkpoint_path=None):
    # Returns target_probs of shape (len(df_in), 169).
    n = len(df_in)
    target_probs = np.zeros((n, 169), dtype=np.float32)

    # Resume from checkpoint if it exists
    done_papers = set()
    if checkpoint_path and os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            for line in f:
                try:
                    e = json.loads(line)
                    pid = int(e['paper_idx'])
                    for tid, p in e['target_scores'].items():
                        if tid in TARGET_IDS:
                            target_probs[pid, TARGET_IDS.index(tid)] = float(p)
                    done_papers.add(pid)
                except Exception: pass
        if done_papers:
            print(f'  [{label}] resuming — {len(done_papers)}/{n} papers already done')

    pending = [i for i in range(n) if i not in done_papers]
    if not pending:
        return target_probs

    f_ckpt = open(checkpoint_path, 'a') if checkpoint_path else None
    lock = Lock()
    t0 = time.time()
    completed = [len(done_papers)]

    def work(paper_idx):
        confident = [g for g in range(1, 18)
                     if goal_probs[paper_idx, g-1] >= threshold]
        if not confident:
            return paper_idx, {}
        row = df_in.iloc[paper_idx]
        scores = call_llm_targets_batched(row, confident)
        with lock:
            if f_ckpt:
                f_ckpt.write(json.dumps({
                    'paper_idx': int(paper_idx),
                    'confident_goals': confident,
                    'target_scores': scores,
                }) + '\n')
                f_ckpt.flush()
            completed[0] += 1
            done_count = completed[0]
            if done_count % max(1, n // 30) == 0 or done_count == n:
                elapsed = time.time() - t0
                fresh = done_count - len(done_papers)
                rate = fresh / elapsed if elapsed > 0 else 0
                eta = (n - done_count) / rate if rate > 0 else 0
                print(f'  [{label}] {done_count}/{n}  {elapsed:.0f}s  '
                      f'{rate:.1f} papers/s  ETA {eta:.0f}s')
        return paper_idx, scores

    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [ex.submit(work, i) for i in pending]
        for fut in as_completed(futures):
            paper_idx, scores = fut.result()
            for tid, p in scores.items():
                if tid in TARGET_IDS:
                    target_probs[paper_idx, TARGET_IDS.index(tid)] = p

    if f_ckpt: f_ckpt.close()
    return target_probs

# ── Run Stage 2 on val + test ────────────────────────────────────────────────
TARGET_CACHE_TEST = os.path.join(RESULTS_DIR, f'target_probs_test_{VERSION}.npy')
TARGET_CACHE_VAL  = os.path.join(RESULTS_DIR, f'target_probs_val_{VERSION}.npy')
STAGE2_CKPT_TEST  = os.path.join(RESULTS_DIR, f'stage2_checkpoint_test_{VERSION}.jsonl')
STAGE2_CKPT_VAL   = os.path.join(RESULTS_DIR, f'stage2_checkpoint_val_{VERSION}.jsonl')

if os.path.exists(TARGET_CACHE_TEST) and os.path.exists(TARGET_CACHE_VAL):
    target_probs_test = np.load(TARGET_CACHE_TEST)
    target_probs_val  = np.load(TARGET_CACHE_VAL)
    print(f'Loaded cached target probs — test {target_probs_test.shape}  '
          f'val {target_probs_val.shape}')
else:
    t0 = time.time()
    print(f'Stage 2 on val ({len(df_val)} papers)...')
    target_probs_val = run_stage2(df_val, goal_probs_val,
                                  checkpoint_path=STAGE2_CKPT_VAL,
                                  label='stage2-val')
    np.save(TARGET_CACHE_VAL, target_probs_val)
    print(f'\nStage 2 on test ({len(df_test)} papers)...')
    target_probs_test = run_stage2(df_test, goal_probs_test,
                                   checkpoint_path=STAGE2_CKPT_TEST,
                                   label='stage2-test')
    np.save(TARGET_CACHE_TEST, target_probs_test)
    # Also write to shared cache so _kg and _doc notebooks can reuse
    np.save(os.path.join(SHARED_LLM_CACHE_DIR, 'target_probs_test.npy'), target_probs_test)
    np.save(os.path.join(SHARED_LLM_CACHE_DIR, 'target_probs_val.npy'), target_probs_val)
    print(f'  Also saved to {SHARED_LLM_CACHE_DIR}/target_probs_*.npy for downstream notebooks')
    print(f'\nStage 2 done in {time.time()-t0:.0f}s. '
          f'Target probs shape: {target_probs_test.shape}')


---
## 7. Propagation through each graph, then compare

In [ ]:
# ── Defensive shape checks ──
# This catches the common failure where caches from a previous N_TEST get loaded
# but the labels were rebuilt for a different N_TEST.
assert goal_probs_val.shape  == val_labels_goal.shape,  \
    f'val shape mismatch: probs={goal_probs_val.shape} labels={val_labels_goal.shape}'
assert goal_probs_test.shape == test_labels_goal.shape, \
    f'test shape mismatch: probs={goal_probs_test.shape} labels={test_labels_goal.shape}'
assert target_probs_val.shape  == val_labels_target.shape,  \
    f'target val mismatch: probs={target_probs_val.shape} labels={val_labels_target.shape}'
assert target_probs_test.shape == test_labels_target.shape, \
    f'target test mismatch: probs={target_probs_test.shape} labels={test_labels_target.shape}'
print(f'Shape check OK: val={goal_probs_val.shape}  test={goal_probs_test.shape}  '
      f'target_val={target_probs_val.shape}  target_test={target_probs_test.shape}')

# === Step 7: propagate LLM target probabilities through each matrix ===
# No new LLM calls. Take the (N, 169) target_probs from Stage 2 and run
# signed-gated propagation through each of the four matrices independently.
# Matrix D is special: its weight w_B is searched jointly with α and conf
# on the val set, then the single best (α, conf, w_B) is reported.

# Same math as v5 (signed-gated propagation), parameterised by graph size.

def propagate_gated_signed(probs, W, alpha=0.8, conf=0.6):
    Wp = np.where(W > 0, W, 0).copy()
    Wn = np.abs(np.where(W < 0, W, 0)).copy()
    rp = Wp.sum(1, keepdims=True); rp[rp==0] = 1.0; Wp /= rp
    rn = Wn.sum(1, keepdims=True); rn[rn==0] = 1.0; Wn /= rn
    gate = (probs >= conf).astype(float)
    gated = probs * gate
    return np.clip(alpha*probs + (1-alpha)*(gated @ Wp.T) - (1-alpha)*(gated @ Wn.T), 0, 1)

def aggregate_targets_to_goals(target_probs):
    # Aggregate 169-dim target probs to 17-dim goal probs via max over each goal.
    N = target_probs.shape[0]
    goal_probs = np.zeros((N, 17))
    for g in range(1, 18):
        t_idxs = GOAL_TO_TARGETS[g]
        goal_probs[:, g-1] = target_probs[:, t_idxs].max(axis=1)
    return goal_probs

def optimise_threshold_per_class(probs, labels, grid=np.arange(0.1, 0.9, 0.05)):
    # Per-class F1-optimal threshold.
    best = np.full(probs.shape[1], 0.5)
    for j in range(probs.shape[1]):
        bf = 0
        for t in grid:
            f = f1_score(labels[:,j], (probs[:,j]>=t).astype(int), zero_division=0)
            if f > bf: bf, best[j] = f, t
    return best

# ── Baseline: no-graph LLM predictions at goal level ────────────────────────
thr_nograph = optimise_threshold_per_class(goal_probs_val, val_labels_goal)
preds_nograph = (goal_probs_test >= thr_nograph[np.newaxis,:]).astype(int)
f_nograph = f1_score(test_labels_goal, preds_nograph, average='macro', zero_division=0)
print(f'No-graph baseline (goal-level LLM only): macro-F1 = {f_nograph:.4f}')

goal_from_target_val  = aggregate_targets_to_goals(target_probs_val)
goal_from_target_test = aggregate_targets_to_goals(target_probs_test)
thr_target_nograph = optimise_threshold_per_class(goal_from_target_val, val_labels_goal)
preds_target_nograph = (goal_from_target_test >= thr_target_nograph[np.newaxis,:]).astype(int)
f_target_nograph = f1_score(test_labels_goal, preds_target_nograph, average='macro', zero_division=0)
print(f'Target-LLM aggregated (no graph):         macro-F1 = {f_target_nograph:.4f}')

# ── Grid search: each graph × alpha × conf (× w for D) ──────────────────────
all_results = []

def eval_goal_graph(name, W_goal, alpha_grid, conf_grid):
    # Graph at goal level — input is goal_probs.
    results = []
    for a in alpha_grid:
        for c in conf_grid:
            pv = propagate_gated_signed(goal_probs_val,  W_goal, alpha=a, conf=c)
            pt = propagate_gated_signed(goal_probs_test, W_goal, alpha=a, conf=c)
            th = optimise_threshold_per_class(pv, val_labels_goal)
            fv = f1_score(val_labels_goal,  (pv >= th).astype(int), average='macro', zero_division=0)
            ft = f1_score(test_labels_goal, (pt >= th).astype(int), average='macro', zero_division=0)
            results.append({'graph': name, 'alpha': a, 'conf': c, 'w_B': None,
                           'val_f1': fv, 'test_f1': ft, 'probs_test': pt, 'thr': th})
    return results

def eval_target_graph(name, W_target, alpha_grid, conf_grid):
    # Graph at target level — input is target_probs, output aggregated to goals.
    results = []
    for a in alpha_grid:
        for c in conf_grid:
            pv_t = propagate_gated_signed(target_probs_val,  W_target, alpha=a, conf=c)
            pt_t = propagate_gated_signed(target_probs_test, W_target, alpha=a, conf=c)
            pv_g = aggregate_targets_to_goals(pv_t)
            pt_g = aggregate_targets_to_goals(pt_t)
            th = optimise_threshold_per_class(pv_g, val_labels_goal)
            fv = f1_score(val_labels_goal,  (pv_g >= th).astype(int), average='macro', zero_division=0)
            ft = f1_score(test_labels_goal, (pt_g >= th).astype(int), average='macro', zero_division=0)
            th_t = optimise_threshold_per_class(pv_t, val_labels_target)
            fv_t = f1_score(val_labels_target,  (pv_t >= th_t).astype(int), average='macro', zero_division=0)
            ft_t = f1_score(test_labels_target, (pt_t >= th_t).astype(int), average='macro', zero_division=0)
            results.append({'graph': name, 'alpha': a, 'conf': c, 'w_B': None,
                           'val_f1': fv, 'test_f1': ft,
                           'val_f1_target': fv_t, 'test_f1_target': ft_t,
                           'probs_test': pt_g, 'probs_test_target': pt_t, 'thr': th})
    return results

def eval_combined_graph(name, alpha_grid, conf_grid, w_grid):
    # Combined target-level graph D = w·B + (1-w)·C. Search over w as well.
    results = []
    for w in w_grid:
        W_D = build_combined_matrix(w)
        for a in alpha_grid:
            for c in conf_grid:
                pv_t = propagate_gated_signed(target_probs_val,  W_D, alpha=a, conf=c)
                pt_t = propagate_gated_signed(target_probs_test, W_D, alpha=a, conf=c)
                pv_g = aggregate_targets_to_goals(pv_t)
                pt_g = aggregate_targets_to_goals(pt_t)
                th = optimise_threshold_per_class(pv_g, val_labels_goal)
                fv = f1_score(val_labels_goal,  (pv_g >= th).astype(int), average='macro', zero_division=0)
                ft = f1_score(test_labels_goal, (pt_g >= th).astype(int), average='macro', zero_division=0)
                th_t = optimise_threshold_per_class(pv_t, val_labels_target)
                fv_t = f1_score(val_labels_target,  (pv_t >= th_t).astype(int), average='macro', zero_division=0)
                ft_t = f1_score(test_labels_target, (pt_t >= th_t).astype(int), average='macro', zero_division=0)
                results.append({'graph': name, 'alpha': a, 'conf': c, 'w_B': w,
                               'val_f1': fv, 'test_f1': ft,
                               'val_f1_target': fv_t, 'test_f1_target': ft_t,
                               'probs_test': pt_g, 'probs_test_target': pt_t, 'thr': th})
    return results

W_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f'\nGrid-searching {len(GRAPHS)} graphs × {len(ALPHA_GRID)} alphas × '
      f'{len(CONF_GRID)} confs (D also × {len(W_GRID)} w values)...')
for key, (name, W, dim) in GRAPHS.items():
    if key.startswith('D_'):
        all_results.extend(eval_combined_graph(key, ALPHA_GRID, CONF_GRID, W_GRID))
    elif dim == 17:
        all_results.extend(eval_goal_graph(key, W, ALPHA_GRID, CONF_GRID))
    else:
        all_results.extend(eval_target_graph(key, W, ALPHA_GRID, CONF_GRID))
    print(f'  {key}: done')

print(f'\nTotal configurations: {len(all_results)}')


---
## 8. Results — seven-matrix comparison

In [ ]:
# Find the best (alpha, conf, [w_B]) per graph using VAL F1, then report TEST F1.

from statsmodels.stats.contingency_tables import mcnemar as _mcnemar_fn

best_per_graph = {}
for key in GRAPHS:
    subset = [r for r in all_results if r['graph'] == key]
    subset.sort(key=lambda r: -r['val_f1'])
    best_per_graph[key] = subset[0]

print('='*95)
print(f'Four-way graph comparison — {N_TEST} test docs, {N_VAL} val, '
      f'avg {test_labels_goal.sum(1).mean():.2f} goals/paper')
print('='*95)
print(f'  Baseline (goal-LLM, no graph):         F1 = {f_nograph:.4f}')
print(f'  Baseline (target-LLM agg, no graph):   F1 = {f_target_nograph:.4f}')
print()
print(f'  {"Graph":36s} {"alpha":>6s} {"conf":>6s} {"w_B":>6s} {"val F1":>8s} {"test F1":>9s} {"Δ vs goal":>11s}')
print('  ' + '-'*92)
for key, name_full_dim in GRAPHS.items():
    name, _, _ = name_full_dim
    r = best_per_graph[key]
    delta = r['test_f1'] - f_nograph
    marker = ' ✓' if delta > 0.005 else ('  ' if -0.005 <= delta <= 0.005 else ' ✗')
    w_str = f'{r["w_B"]:.2f}' if r['w_B'] is not None else '—'
    print(f'  {name:36s} {r["alpha"]:>6.2f} {r["conf"]:>6.2f} {w_str:>6s} '
          f'{r["val_f1"]:>8.4f} {r["test_f1"]:>9.4f} {delta:>+10.4f}{marker}')

# ── McNemar: each graph vs goal-LLM baseline ────────────────────────────────
print('\n── McNemar significance tests (each graph vs goal-LLM no-graph baseline) ──')

def run_mcnemar(y, y_a, y_b):
    fa, fb, yt = y_a.flatten(), y_b.flatten(), y.flatten()
    b = int(((fa == yt) & (fb != yt)).sum())
    c = int(((fa != yt) & (fb == yt)).sum())
    if b + c == 0: return None, b, c
    res = _mcnemar_fn([[0, b], [c, 0]], exact=(b+c < 25), correction=True)
    return res.pvalue, b, c

for key in GRAPHS:
    r = best_per_graph[key]
    preds_graph = (r['probs_test'] >= r['thr']).astype(int)
    p, b, c = run_mcnemar(test_labels_goal, preds_nograph, preds_graph)
    if p is None:
        sig_str = '(no disagreements)'
    else:
        sig_str = '*** significant' if p < 0.05 else 'not significant'
    p_str = f'{p:.4f}' if p is not None else 'N/A'
    print(f'  {GRAPHS[key][0]:36s}  p = {p_str}  {sig_str}')

# ── Target-level F1 (for 169x169 graphs only) ───────────────────────────────
print('\n── Target-level macro-F1 (169 classes; only for target-level graphs) ──')
for key in GRAPHS:
    _, _, dim = GRAPHS[key]
    if dim != 169: continue
    r = best_per_graph[key]
    if 'test_f1_target' in r:
        print(f'  {GRAPHS[key][0]:36s}  target-F1 = {r["test_f1_target"]:.4f}')

# ── Per-SDG breakdown for the overall best graph ────────────────────────────
print('\n── Per-SDG delta (best graph − goal-LLM baseline), top 5 movers ──')
overall_best_key = max(GRAPHS.keys(), key=lambda k: best_per_graph[k]['val_f1'])
r_best = best_per_graph[overall_best_key]
preds_best = (r_best['probs_test'] >= r_best['thr']).astype(int)
print(f'  Best graph by val F1: {GRAPHS[overall_best_key][0]}')
deltas = []
for j in range(17):
    fb = f1_score(test_labels_goal[:,j], preds_nograph[:,j], zero_division=0)
    fg = f1_score(test_labels_goal[:,j], preds_best[:,j],    zero_division=0)
    deltas.append((j+1, SDG_NAMES[j+1], fb, fg, fg-fb))
for g, name, fb, fg, d in sorted(deltas, key=lambda x: -abs(x[4]))[:5]:
    print(f'    SDG {g:2d} {name:25s}  {fb:.4f} → {fg:.4f}  ({d:+.4f})')

# ── Save results CSV ────────────────────────────────────────────────────────
summary_rows = []
summary_rows.append({'graph': 'goal-LLM (no graph)', 'alpha': None, 'conf': None,
                    'w_B': None, 'val_f1': None, 'test_f1': f_nograph})
summary_rows.append({'graph': 'target-LLM aggregated (no graph)', 'alpha': None, 'conf': None,
                    'w_B': None, 'val_f1': None, 'test_f1': f_target_nograph})
for key in GRAPHS:
    r = best_per_graph[key]
    summary_rows.append({'graph': GRAPHS[key][0], 'alpha': r['alpha'], 'conf': r['conf'],
                        'w_B': r['w_B'], 'val_f1': r['val_f1'], 'test_f1': r['test_f1']})

df_summary = pd.DataFrame(summary_rows)
out_csv = os.path.join(RESULTS_DIR, f'graph_comparison_{VERSION}.csv')
df_summary.to_csv(out_csv, index=False)
print(f'\nSaved: {out_csv}')
print(df_summary.to_string(index=False))


In [ ]:
import os
results_dir = os.path.join(DRIVE_ROOT, 'results_v5_multi_gemma4-26b_n10000')
if os.path.exists(results_dir):
    files = sorted(os.listdir(results_dir))
    print(f'{len(files)} files in {results_dir}:')
    for f in files:
        size_mb = os.path.getsize(os.path.join(results_dir, f)) / 1e6
        print(f'  {size_mb:7.2f} MB  {f}')
else:
    print(f'NOT FOUND: {results_dir}')

---

In [ ]:
!pip uninstall -y torchcodec

---
## 9. Per-document methods (doc_neighbours + doc_entities)

Reuses the in-memory `goal_probs_*` and `target_probs_*` tensors. 5k neighbour pool, K=20, per-goal λ with 5-fold CV. v3 fixes: pool cache is SEED-keyed and length-checked; v2 entity cache is length-checked before reuse; doc-side DOI fingerprint printed for cross-model comparison.


In [ ]:
# ── Doc-methods config ──
ENABLE_DOC_ENTITIES = True
DOC_NEIGHBOURS_K    = 20
N_NEIGHBOUR_POOL    = 5000
PER_GOAL_LAMBDA     = True
CV_FOLDS_LAMBDA     = 5

# ── Build the 5k neighbour candidate pool (disjoint from fewshot/val/test) ──
used_dois = set(df_fewshot['doi']) | set(df_val['doi']) | set(df_test['doi'])
df_pool_candidates = df_multi[~df_multi['doi'].isin(used_dois)]
print(f'Neighbour-pool candidates available: {len(df_pool_candidates)}')
if len(df_pool_candidates) >= N_NEIGHBOUR_POOL:
    df_pool = df_pool_candidates.sample(N_NEIGHBOUR_POOL, random_state=SEED).reset_index(drop=True)
else:
    df_pool = df_pool_candidates.reset_index(drop=True)

POOL_ABSTRACTS_CACHE = os.path.join(DATA_CACHE,
    f'neighbour_pool_abstracts_n{N_NEIGHBOUR_POOL}_seed{SEED}.parquet')

_pool_ok = False
if os.path.exists(POOL_ABSTRACTS_CACHE):
    _candidate = pd.read_parquet(POOL_ABSTRACTS_CACHE)
    # Accept the cache if its length is within 90% of the requested pool size
    # (some DOIs always fail OpenAlex; 100% match is unrealistic).
    if 0.9 * N_NEIGHBOUR_POOL <= len(_candidate) <= 1.1 * N_NEIGHBOUR_POOL:
        df_pool = _candidate
        print(f'Loaded cached neighbour-pool abstracts: {len(df_pool)} papers (within 10% of requested {N_NEIGHBOUR_POOL})')
        _pool_ok = True
    else:
        print(f'Neighbour-pool cache size mismatch: {len(_candidate)} vs requested {N_NEIGHBOUR_POOL}. Rebuilding.')
        os.remove(POOL_ABSTRACTS_CACHE)

if not _pool_ok:
    print(f'Fetching abstracts for {len(df_pool)} pool papers from OpenAlex...')
    pool_dois = df_pool['doi'].astype(str).str.strip().str.lower().tolist()
    pool_abstract_dict = fetch_abstracts_openalex(pool_dois)
    df_pool = df_pool.copy()
    df_pool['_doi_lower'] = df_pool['doi'].astype(str).str.strip().str.lower()
    df_pool['title']    = df_pool['_doi_lower'].map(lambda d: (pool_abstract_dict.get(d) or {}).get('title', ''))
    df_pool['abstract'] = df_pool['_doi_lower'].map(lambda d: (pool_abstract_dict.get(d) or {}).get('abstract', ''))
    df_pool = df_pool.drop(columns=['_doi_lower'])
    df_pool = df_pool[df_pool['abstract'].fillna('').str.len() > 50].reset_index(drop=True)
    df_pool.to_parquet(POOL_ABSTRACTS_CACHE, index=False)
    print(f'Saved {len(df_pool)} neighbour-pool papers to {POOL_ABSTRACTS_CACHE}')

pool_labels_goal = df_pool[goal_cols].values.astype(np.float32)
print(f'Final neighbour pool: {len(df_pool)} docs')

import hashlib as _hl
_dois_test_now = df_test['doi'].astype(str).tolist()
_doc_hash = _hl.sha1(''.join(_dois_test_now).encode('utf-8')).hexdigest()[:12]
print(f'\n── Doc-methods fingerprint: test hash={_doc_hash} (compare to Cell 9) ──')

# Pull SBERT into scope
from sentence_transformers import SentenceTransformer

def opt_threshold(probs, labels):
    return optimise_threshold_per_class(probs, labels)
def macro_f1(probs, labels, thr):
    return f1_score(labels, (probs >= thr).astype(int), average='macro', zero_division=0)

# ── Per-document method evaluation ───────────
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# ── Reusable helpers ────────────────────────────────────────────────────────
def aggregate_targets_to_goals(target_probs):
    N = target_probs.shape[0]
    out = np.zeros((N, 17))
    for g in range(1, 18):
        out[:, g-1] = target_probs[:, GOAL_TO_TARGETS[g]].max(axis=1)
    return out

def opt_threshold(probs, labels, grid=np.arange(0.1, 0.9, 0.05)):
    best = np.full(probs.shape[1], 0.5)
    for j in range(probs.shape[1]):
        bf = 0
        for t in grid:
            f = f1_score(labels[:,j], (probs[:,j]>=t).astype(int), zero_division=0)
            if f > bf: bf, best[j] = f, t
    return best

def macro_f1(probs, labels, thr):
    return f1_score(labels, (probs >= thr).astype(int), average='macro', zero_division=0)

# ── Baselines ───────────────────────────────────────────────────────────────
goal_baseline_thr = opt_threshold(goal_probs_val, val_labels_goal)
f_goal_baseline = macro_f1(goal_probs_test, test_labels_goal, goal_baseline_thr)

target_aggregated_val  = aggregate_targets_to_goals(target_probs_val)
target_aggregated_test = aggregate_targets_to_goals(target_probs_test)
target_thr        = opt_threshold(target_aggregated_val, val_labels_goal)
f_target_baseline = macro_f1(target_aggregated_test, test_labels_goal, target_thr)

print('Baselines (LLM):')
print(f'  goal-LLM (no graph):              {f_goal_baseline:.4f}')
print(f'  target-LLM aggregated (CEILING):  {f_target_baseline:.4f}')

# ── TF-IDF + LogReg baseline (no LLM, no graph) ────────────────────────────
# Cheap reference point: what does a vanilla bag-of-words text classifier do?
# Helps a reader judge whether 0.54 macro-F1 is good or just average.
print('\nTF-IDF baseline (one-vs-rest LogReg on abstracts)...')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# Train on fewshot+val, evaluate on test. (Fewshot is small but it's all the
# labelled data this baseline is allowed to peek at; LLM uses it for prompting.)
_train_df = pd.concat([df_fewshot, df_val], ignore_index=True)
_train_text = [str(r.get('title','')) + '. ' + str(r.get('abstract',''))
               for _, r in _train_df.iterrows()]
_test_text  = [str(r.get('title','')) + '. ' + str(r.get('abstract',''))
               for _, r in df_test.iterrows()]
_train_y = _train_df[goal_cols].values.astype(int)

_vec = TfidfVectorizer(min_df=2, max_df=0.9, ngram_range=(1,2),
                       max_features=20000, stop_words='english')
_Xtr = _vec.fit_transform(_train_text)
_Xte = _vec.transform(_test_text)
_clf = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=1.0,
                                                class_weight='balanced'))
_clf.fit(_Xtr, _train_y)
_tfidf_probs_test = _clf.predict_proba(_Xte)
# Tune per-goal threshold on val (refit-with-val approach: reuse opt_threshold
# logic by predicting val probs from a model trained on fewshot only).
_vec_fs = TfidfVectorizer(min_df=2, max_df=0.9, ngram_range=(1,2),
                          max_features=20000, stop_words='english')
_Xfs   = _vec_fs.fit_transform([str(r.get('title','')) + '. ' + str(r.get('abstract',''))
                                  for _, r in df_fewshot.iterrows()])
_Xval  = _vec_fs.transform([str(r.get('title','')) + '. ' + str(r.get('abstract',''))
                              for _, r in df_val.iterrows()])
_clf_fs = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=1.0,
                                                    class_weight='balanced'))
_clf_fs.fit(_Xfs, df_fewshot[goal_cols].values.astype(int))
_tfidf_probs_val = _clf_fs.predict_proba(_Xval)
_tfidf_thr = opt_threshold(_tfidf_probs_val, val_labels_goal)
f_tfidf = macro_f1(_tfidf_probs_test, test_labels_goal, _tfidf_thr)
print(f'  TF-IDF + LogReg:                  {f_tfidf:.4f}')

# ── Predict-marginal baseline ──────────────────────────────────────────────
# Predicts each goal's val frequency for every test doc — a bare-floor sanity
# check. If a method ties this, the method has zero discriminative power.
_marginals = val_labels_goal.mean(axis=0)
_marg_probs_test = np.tile(_marginals, (len(test_labels_goal), 1))
_marg_thr = opt_threshold(np.tile(_marginals, (len(val_labels_goal), 1)),
                            val_labels_goal)
f_marginal = macro_f1(_marg_probs_test, test_labels_goal, _marg_thr)
print(f'  Predict-marginal (floor):         {f_marginal:.4f}')

# ── doc_neighbours ──────────────────────────────────────────────────────
print(f'\nLoading SBERT for doc_neighbours (pool={len(df_pool)}, K={DOC_NEIGHBOURS_K})...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

def encode_abstracts(text_list):
    return sbert.encode([t[:1500] for t in text_list], show_progress_bar=False,
                        normalize_embeddings=True, batch_size=64)

pool_text = [str(r.get('title','')) + '. ' + str(r.get('abstract',''))
             for _, r in df_pool.iterrows()]
val_text  = [str(r.get('title','')) + '. ' + str(r.get('abstract',''))
             for _, r in df_val.iterrows()]
test_text = [str(r.get('title','')) + '. ' + str(r.get('abstract',''))
             for _, r in df_test.iterrows()]

print(f'  Encoding {len(pool_text)} pool + {len(val_text)} val + {len(test_text)} test...')
pool_embs = encode_abstracts(pool_text)
val_embs  = encode_abstracts(val_text)
test_embs = encode_abstracts(test_text)

def doc_neighbours_evidence(query_embs, K=DOC_NEIGHBOURS_K):
    """For each query, similarity-weighted neighbour label vote from the pool."""
    n = len(query_embs)
    out = np.zeros((n, 17), dtype=np.float32)
    for i in range(n):
        sims = pool_embs @ query_embs[i]
        topk = np.argsort(-sims)[:K]
        sim_weights = np.clip(sims[topk], 0, None)
        if sim_weights.sum() < 1e-6:
            out[i] = pool_labels_goal[topk].mean(axis=0)
        else:
            out[i] = (pool_labels_goal[topk].T @ sim_weights) / sim_weights.sum()
    return out

print('  Computing neighbour evidence...')
nbr_val_evidence  = doc_neighbours_evidence(val_embs)
nbr_test_evidence = doc_neighbours_evidence(test_embs)

# Per-goal λ calibration
print(f'  Calibrating λ {"per-goal" if PER_GOAL_LAMBDA else "globally"} on val...')
def calibrate_per_goal_lambda(base, evidence, labels, grid=np.linspace(0, 0.7, 15)):
    """For each goal independently, find λ that maximises F1 on val."""
    n_goals = base.shape[1]
    lambdas = np.zeros(n_goals)
    for g in range(n_goals):
        best_f, best_l = -1, 0
        for lam in grid:
            mixed = np.clip(lam * evidence[:, g] + (1-lam) * base[:, g], 0, 1)
            for t in np.arange(0.1, 0.9, 0.05):
                f = f1_score(labels[:, g], (mixed >= t).astype(int), zero_division=0)
                if f > best_f:
                    best_f, best_l = f, lam
        lambdas[g] = best_l
    return lambdas

def calibrate_per_goal_lambda_cv(base, evidence, labels, k=5,
                                  grid=np.linspace(0, 0.7, 15), seed=SEED):
    """K-fold CV variant — picks the λ with highest mean held-out F1 per goal,
    rather than the λ that overfits the full val set. Reduces variance from the
    240-combination-per-goal search on only ~200 val docs.

    Falls back to the non-CV version if k <= 1.
    """
    if k is None or k <= 1:
        return calibrate_per_goal_lambda(base, evidence, labels, grid=grid)
    rng = np.random.default_rng(seed)
    n = base.shape[0]
    idx = np.arange(n); rng.shuffle(idx)
    folds = np.array_split(idx, k)
    n_goals = base.shape[1]
    lambdas = np.zeros(n_goals)
    for g in range(n_goals):
        # mean held-out F1 per λ across folds; threshold tuned on each train fold
        per_lambda_scores = []
        for lam in grid:
            fold_scores = []
            for f_idx in range(k):
                hold = folds[f_idx]
                train = np.concatenate([folds[j] for j in range(k) if j != f_idx])
                mixed_train = np.clip(lam * evidence[train, g]
                                       + (1-lam) * base[train, g], 0, 1)
                mixed_hold  = np.clip(lam * evidence[hold, g]
                                       + (1-lam) * base[hold, g], 0, 1)
                # pick threshold on train fold
                best_t, best_t_f = 0.5, -1
                for t in np.arange(0.1, 0.9, 0.05):
                    ft = f1_score(labels[train, g], (mixed_train >= t).astype(int),
                                  zero_division=0)
                    if ft > best_t_f:
                        best_t_f, best_t = ft, t
                # evaluate on held-out
                fold_scores.append(
                    f1_score(labels[hold, g], (mixed_hold >= best_t).astype(int),
                             zero_division=0)
                )
            per_lambda_scores.append(np.mean(fold_scores))
        lambdas[g] = grid[int(np.argmax(per_lambda_scores))]
    return lambdas

if PER_GOAL_LAMBDA:
    # Use CV variant if CV_FOLDS_LAMBDA > 1 (set in cell 2). Falls back to
    # the original argmax-on-full-val behaviour when CV_FOLDS_LAMBDA = 1.
    lambdas_nbr = calibrate_per_goal_lambda_cv(target_aggregated_val,
                                                nbr_val_evidence,
                                                val_labels_goal,
                                                k=CV_FOLDS_LAMBDA)
    print(f'  Per-goal λ_nbr (all 17): '
          f'{[(g+1, round(lambdas_nbr[g], 2)) for g in range(17)]}')
    nbr_val_combined  = np.clip(lambdas_nbr[None, :] * nbr_val_evidence
                                + (1 - lambdas_nbr[None, :]) * target_aggregated_val, 0, 1)
    nbr_test_combined = np.clip(lambdas_nbr[None, :] * nbr_test_evidence
                                + (1 - lambdas_nbr[None, :]) * target_aggregated_test, 0, 1)
else:
    best_lambda, best_val_f1 = 0.0, -1.0
    for lam in np.linspace(0, 0.7, 15):
        v = np.clip(lam * nbr_val_evidence + (1-lam) * target_aggregated_val, 0, 1)
        thr = opt_threshold(v, val_labels_goal)
        f = macro_f1(v, val_labels_goal, thr)
        if f > best_val_f1:
            best_val_f1, best_lambda = f, lam
    print(f'  Global λ_nbr: {best_lambda:.2f}  (val F1={best_val_f1:.4f})')
    nbr_val_combined  = np.clip(best_lambda * nbr_val_evidence
                                + (1-best_lambda) * target_aggregated_val, 0, 1)
    nbr_test_combined = np.clip(best_lambda * nbr_test_evidence
                                + (1-best_lambda) * target_aggregated_test, 0, 1)

doc_nbr_thr  = opt_threshold(nbr_val_combined, val_labels_goal)
f_doc_nbr    = macro_f1(nbr_test_combined, test_labels_goal, doc_nbr_thr)
val_f1_nbr   = macro_f1(nbr_val_combined, val_labels_goal, doc_nbr_thr)
print(f'  doc_neighbours (test): {f_doc_nbr:.4f}    (val: {val_f1_nbr:.4f})')

# ── doc_entities ──────────────────────────────
f_doc_ent = None
ent_evidence_test = ent_evidence_val = None
if ENABLE_DOC_ENTITIES:
    print('\ndoc_entities — looking up DOI-hash-keyed cache...')
    # Entities are model-agnostic to first order (same abstract → same
    # countries / methods / materials regardless of which LLM extracted them),
    # so this cache lives in data_cache/ and is shared across all 4 models in
    # the registry. First run extracts; subsequent model runs reuse for free.
    import hashlib as _hl
    _test_dois = df_test['doi'].astype(str).tolist()
    _val_dois  = df_val['doi'].astype(str).tolist()
    _test_hash = _hl.sha1(''.join(_test_dois).encode('utf-8')).hexdigest()[:12]
    _val_hash  = _hl.sha1(''.join(_val_dois ).encode('utf-8')).hexdigest()[:12]
    ENT_CACHE_TEST = os.path.join(DATA_CACHE, f'entities_test_hash_{_test_hash}.json')
    ENT_CACHE_VAL  = os.path.join(DATA_CACHE, f'entities_val_hash_{_val_hash}.json')
    print(f'  Hash-keyed cache paths:')
    print(f'    test: {ENT_CACHE_TEST}  ({"hit" if os.path.exists(ENT_CACHE_TEST) else "miss"})')
    print(f'    val:  {ENT_CACHE_VAL}   ({"hit" if os.path.exists(ENT_CACHE_VAL) else "miss"})')

    _ent_ok = False
    if os.path.exists(ENT_CACHE_TEST) and os.path.exists(ENT_CACHE_VAL):
        with open(ENT_CACHE_TEST) as f: test_entities = json.load(f)
        with open(ENT_CACHE_VAL)  as f: val_entities  = json.load(f)
        if len(test_entities) == len(df_test) and len(val_entities) == len(df_val):
            print(f'  Loaded hash-keyed cache: {len(test_entities)} test + {len(val_entities)} val entities')
            print(f'    (DOI alignment guaranteed by filename hash)')
            _ent_ok = True
        else:
            print(f'  Hash-keyed cache exists but length mismatch — re-extracting.')

    if not _ent_ok:
        print(f'  Running fresh Stage 1.5 entity extraction on {len(df_test)+len(df_val)} abstracts...')
        print(f'  (~5 min on Gemma at CONCURRENCY={CONCURRENCY}; subsequent model runs will reuse this cache for free)')

        def build_entity_prompt(row):
            title = str(row.get('title',''))[:200]
            abstract = str(row.get('abstract',''))[:600]
            return ('Extract scientific entities from this abstract. Output JSON only:\n'
                    '{"entities": ["entity1", "entity2", ...]}\n\n'
                    'Include: countries, organisms, diseases, technologies, methods, '
                    'materials, regions, key concepts. Limit to 12 entities, '
                    'lowercase short noun phrases.\n\n'
                    f'TITLE: {title}\nABSTRACT: {abstract}')

        def extract_entities(row):
            for _ in range(2):
                try:
                    extra_body = {}
                    ctk = _CFG.get('chat_template_kwargs')
                    if ctk:
                        if THINKING_MODE and 'enable_thinking' in ctk:
                            ctk = {**ctk, 'enable_thinking': True}
                        extra_body['chat_template_kwargs'] = ctk
                    resp = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{'role':'user', 'content': build_entity_prompt(row)}],
                        max_tokens=200, temperature=0.0,
                        extra_body=extra_body or None,
                    )
                    raw = resp.choices[0].message.content or ''
                    if _CFG.get('strip_think_blocks'):
                        raw = _strip_think(raw)
                    raw = raw.strip()
                    m = re.search(r'\{.*\}', raw, re.DOTALL)
                    if m:
                        obj = json.loads(m.group(0))
                        ents = obj.get('entities', [])
                        return [str(e).lower().strip() for e in ents if isinstance(e, str)][:12]
                except Exception:
                    pass
            return []

        test_entities = [None] * len(df_test)
        val_entities  = [None] * len(df_val)
        lock = Lock(); completed = [0]; t0 = time.time()
        def work(idx, df_in, store):
            ents = extract_entities(df_in.iloc[idx])
            with lock:
                store[idx] = ents
                completed[0] += 1
                if completed[0] % 200 == 0:
                    rate = completed[0] / max(1, time.time() - t0)
                    print(f'    {completed[0]} done  {rate:.1f} docs/s')
        with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
            futs = [ex.submit(work, i, df_test, test_entities) for i in range(len(df_test))]
            futs += [ex.submit(work, i, df_val,  val_entities)  for i in range(len(df_val))]
            for _ in as_completed(futs): pass

        with open(ENT_CACHE_TEST, 'w') as f: json.dump(test_entities, f)
        with open(ENT_CACHE_VAL,  'w') as f: json.dump(val_entities,  f)
        print(f'  Saved hash-keyed cache:')
        print(f'    {ENT_CACHE_TEST}')
        print(f'    {ENT_CACHE_VAL}')
        print(f'  All 4 models in the registry can now reuse these entity caches.')

    # Build target-keyword profiles
    STOPWORDS_T = set('a an and are as at be by for from has have he her him his '
                      'i in is it its of on or our she that the their them they this '
                      'to was we were will with you your via per through into onto '
                      'upon during such other than then which when where what who '
                      'whom whose why how all some any most much many few also more '
                      'less least best own no not nor only so too very same just both '
                      'each every either neither been being do does did doing having had'.split())
    def _kw(text):
        text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
        return set(t for t in text.split() if len(t) > 2 and t not in STOPWORDS_T)
    target_word_sets = {tid: _kw(TARGET_DESC[tid]) for tid in TARGET_IDS}
    goal_word_sets = {}
    for g in range(1, 18):
        gws = set()
        for tid_idx in GOAL_TO_TARGETS[g]:
            gws |= target_word_sets[TARGET_IDS[tid_idx]]
        goal_word_sets[g] = gws

    def entities_to_goal_evidence(entities):
        if not entities: return np.zeros(17, dtype=np.float32)
        ent_words = set()
        for e in entities:
            ent_words.update(_kw(e))
        ev = np.zeros(17, dtype=np.float32)
        for g in range(1, 18):
            gw = goal_word_sets[g]
            union = gw | ent_words
            if union:
                ev[g-1] = len(gw & ent_words) / len(union)
        if ev.max() > 0: ev = ev / ev.max()
        return ev

    print('  Computing entity evidence vectors...')
    ent_evidence_val  = np.array([entities_to_goal_evidence(e or []) for e in val_entities],
                                  dtype=np.float32)
    ent_evidence_test = np.array([entities_to_goal_evidence(e or []) for e in test_entities],
                                  dtype=np.float32)

    # Per-goal λ for entities
    if PER_GOAL_LAMBDA:
        lambdas_ent = calibrate_per_goal_lambda_cv(target_aggregated_val,
                                                    ent_evidence_val,
                                                    val_labels_goal,
                                                    k=CV_FOLDS_LAMBDA)
        print(f'  Per-goal λ_ent (all 17): '
              f'{[(g+1, round(lambdas_ent[g], 2)) for g in range(17)]}')
        ent_val_combined  = np.clip(lambdas_ent[None, :] * ent_evidence_val
                                     + (1 - lambdas_ent[None, :]) * target_aggregated_val, 0, 1)
        ent_test_combined = np.clip(lambdas_ent[None, :] * ent_evidence_test
                                     + (1 - lambdas_ent[None, :]) * target_aggregated_test, 0, 1)
    else:
        best_lambda_ent, best_v = 0, -1
        for lam in np.linspace(0, 0.6, 13):
            v = np.clip(lam * ent_evidence_val + (1-lam) * target_aggregated_val, 0, 1)
            thr = opt_threshold(v, val_labels_goal)
            f = macro_f1(v, val_labels_goal, thr)
            if f > best_v: best_v, best_lambda_ent = f, lam
        ent_val_combined  = np.clip(best_lambda_ent * ent_evidence_val
                                     + (1-best_lambda_ent) * target_aggregated_val, 0, 1)
        ent_test_combined = np.clip(best_lambda_ent * ent_evidence_test
                                     + (1-best_lambda_ent) * target_aggregated_test, 0, 1)

    doc_ent_thr  = opt_threshold(ent_val_combined, val_labels_goal)
    f_doc_ent    = macro_f1(ent_test_combined, test_labels_goal, doc_ent_thr)
    val_f1_ent   = macro_f1(ent_val_combined, val_labels_goal, doc_ent_thr)
    print(f'  doc_entities (test): {f_doc_ent:.4f}    (val: {val_f1_ent:.4f})')

# ── Three-way combined: LLM + neighbours + entities ────────────────────────
f_combo = None
if f_doc_ent is not None:
    print('\nThree-way combination (LLM + neighbours + entities)...')
    if PER_GOAL_LAMBDA:
        # Per-goal three-way: search (a, b) per goal s.t. a*nbr + b*ent + (1-a-b)*base
        lambdas_a = np.zeros(17); lambdas_b = np.zeros(17)
        grid = np.linspace(0, 0.5, 6)
        for g in range(17):
            best_f, ba, bb = -1, 0, 0
            for a in grid:
                for b in grid:
                    if a + b > 0.8: continue
                    mixed = np.clip(a * nbr_val_evidence[:, g]
                                    + b * ent_evidence_val[:, g]
                                    + (1-a-b) * target_aggregated_val[:, g], 0, 1)
                    for t in np.arange(0.1, 0.9, 0.05):
                        f = f1_score(val_labels_goal[:, g],
                                     (mixed >= t).astype(int), zero_division=0)
                        if f > best_f:
                            best_f, ba, bb = f, a, b
            lambdas_a[g], lambdas_b[g] = ba, bb
        combo_val  = np.clip(lambdas_a[None,:] * nbr_val_evidence
                              + lambdas_b[None,:] * ent_evidence_val
                              + (1 - lambdas_a[None,:] - lambdas_b[None,:]) * target_aggregated_val, 0, 1)
        combo_test = np.clip(lambdas_a[None,:] * nbr_test_evidence
                              + lambdas_b[None,:] * ent_evidence_test
                              + (1 - lambdas_a[None,:] - lambdas_b[None,:]) * target_aggregated_test, 0, 1)
    else:
        best_a, best_b, best_v = 0, 0, -1
        for a in np.linspace(0, 0.5, 6):
            for b in np.linspace(0, 0.5, 6):
                if a + b > 0.8: continue
                v = np.clip(a*nbr_val_evidence + b*ent_evidence_val
                            + (1-a-b)*target_aggregated_val, 0, 1)
                thr = opt_threshold(v, val_labels_goal)
                f = macro_f1(v, val_labels_goal, thr)
                if f > best_v: best_v, best_a, best_b = f, a, b
        combo_val  = np.clip(best_a*nbr_val_evidence + best_b*ent_evidence_val
                              + (1-best_a-best_b)*target_aggregated_val, 0, 1)
        combo_test = np.clip(best_a*nbr_test_evidence + best_b*ent_evidence_test
                              + (1-best_a-best_b)*target_aggregated_test, 0, 1)
        print(f'  Global λ_nbr={best_a:.2f}, λ_ent={best_b:.2f}')
    combo_thr = opt_threshold(combo_val, val_labels_goal)
    f_combo = macro_f1(combo_test, test_labels_goal, combo_thr)
    val_f1_combo = macro_f1(combo_val, val_labels_goal, combo_thr)
    print(f'  Combined (test): {f_combo:.4f}    (val: {val_f1_combo:.4f})')

# ── Final comparison ───────────────────────────────────────────────────────
print('\n' + '='*72)
print('Per-document method comparison (v3)')
print('='*72)
print(f'  Method                                  macro-F1   Δ vs target-baseline')
print(f'  ' + '-'*68)
print(f'  goal-LLM (no graph)                     {f_goal_baseline:.4f}    --')
print(f'  target-LLM aggregated (CEILING)         {f_target_baseline:.4f}    ---')
print(f'  doc_neighbours (pool=5k, K=20)           {f_doc_nbr:.4f}    {(f_doc_nbr-f_target_baseline)*100:+.2f} pp')
if f_doc_ent is not None:
    print(f'  doc_entities                            {f_doc_ent:.4f}    {(f_doc_ent-f_target_baseline)*100:+.2f} pp')
if f_combo is not None:
    print(f'  doc_neighbours + doc_entities (combined) {f_combo:.4f}    {(f_combo-f_target_baseline)*100:+.2f} pp')

# Save CSV
import pandas as pd
summary_rows = [
    {'method': 'goal-LLM (no graph)',                'f1': float(f_goal_baseline)},
    {'method': 'target-LLM aggregated (CEILING)',    'f1': float(f_target_baseline)},
    {'method': f'doc_neighbours (pool={N_NEIGHBOUR_POOL}, K={DOC_NEIGHBOURS_K})',
     'f1': float(f_doc_nbr)},
]
if f_doc_ent is not None:
    summary_rows.append({'method': 'doc_entities (per-goal λ)' if PER_GOAL_LAMBDA else 'doc_entities',
                          'f1': float(f_doc_ent)})
if f_combo is not None:
    summary_rows.append({'method': 'doc_neighbours + doc_entities (combined)',
                          'f1': float(f_combo)})

df_summary = pd.DataFrame(summary_rows)
out_csv = os.path.join(RESULTS_DIR, f'doc_methods_v3_comparison_{VERSION}.csv')
df_summary.to_csv(out_csv, index=False)
print(f'\nSaved: {out_csv}')
print(df_summary.to_string(index=False))

# ── Per-goal F1 breakdown ────────────────────────────────────────
# Macro-F1 hides which goals each method actually helps on. The per-goal table
# below makes the per-goal-λ story falsifiable: if SDGs with strong topical
# clustering really benefit most from neighbour signal, that should show up in
# rows where λ_nbr is high AND Δ_nbr is positive.
print('\n' + '='*88)
print('Per-goal macro-F1 breakdown')
print('='*88)

def _per_goal_f1(probs, labels, thr):
    return f1_score(labels, (probs >= thr).astype(int),
                    average=None, zero_division=0)

per_goal_rows = []
_f1_target  = _per_goal_f1(target_aggregated_test, test_labels_goal, target_thr)
_f1_nbr     = _per_goal_f1(nbr_test_combined,      test_labels_goal, doc_nbr_thr)
_f1_ent     = (_per_goal_f1(ent_test_combined,    test_labels_goal, doc_ent_thr)
               if f_doc_ent is not None else [None]*17)
_f1_combo   = (_per_goal_f1(combo_test,            test_labels_goal, combo_thr)
               if f_combo is not None else [None]*17)
_lam_nbr_arr = lambdas_nbr if PER_GOAL_LAMBDA else np.full(17, np.nan)
_lam_ent_arr = (lambdas_ent if (PER_GOAL_LAMBDA and f_doc_ent is not None)
                 else np.full(17, np.nan))

for g in range(17):
    per_goal_rows.append({
        'goal': g + 1,
        'name': SDG_NAMES[g + 1],
        'f1_target_baseline': float(_f1_target[g]),
        'f1_doc_neighbours':  float(_f1_nbr[g]),
        'delta_nbr_pp':       float((_f1_nbr[g] - _f1_target[g]) * 100),
        'lambda_nbr':         float(_lam_nbr_arr[g]) if not np.isnan(_lam_nbr_arr[g]) else None,
        'f1_doc_entities':    (float(_f1_ent[g]) if _f1_ent[g] is not None else None),
        'lambda_ent':         (float(_lam_ent_arr[g])
                                if not np.isnan(_lam_ent_arr[g]) else None),
        'f1_combined':        (float(_f1_combo[g]) if _f1_combo[g] is not None else None),
    })
df_per_goal = pd.DataFrame(per_goal_rows)
print(df_per_goal.to_string(index=False))
out_per_goal = os.path.join(RESULTS_DIR, f'doc_methods_v3_per_goal_{VERSION}.csv')
df_per_goal.to_csv(out_per_goal, index=False)
print(f'\nSaved per-goal breakdown: {out_per_goal}')

# Augment the existing summary CSV with the two new baselines so the headline
# table tells the full story.
extra_rows = [
    {'method': 'TF-IDF + LogReg (no LLM)',          'f1': float(f_tfidf)},
    {'method': 'Predict-marginal (floor)',          'f1': float(f_marginal)},
]
df_summary_full = pd.concat([pd.DataFrame(extra_rows), df_summary], ignore_index=True)
out_summary_full = os.path.join(RESULTS_DIR, f'doc_methods_v3_comparison_with_baselines_{VERSION}.csv')
df_summary_full.to_csv(out_summary_full, index=False)
print(f'\nFull comparison (incl. baselines): {out_summary_full}')
print(df_summary_full.to_string(index=False))


In [ ]:
# Tail the vLLM server log (throughput / KV-cache / preemption lines).
import re as _re
try:
    with open(VLLM_LOG) as _f:
        _lines = [l.rstrip() for l in _f
                  if _re.search(r'KV cache|throughput|preemp', l)]
    print('\n'.join(_lines[-20:]) or '(no matching lines yet)')
except FileNotFoundError:
    print(f'No vLLM log at {VLLM_LOG} — server not started in this session.')


## 10. Save Configuration


In [ ]:
import sys
import platform
import pkg_resources
import json
import os

def save_config_for_reproducibility(results_dir, version_string):
    config_data = {}

    # 1. Python Environment
    config_data['python_version'] = sys.version
    config_data['os_info'] = platform.platform()

    # 2. Installed Python Packages
    installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    config_data['installed_packages'] = installed_packages

    # 3. Key Notebook Configuration Variables (from Cell 2)
    config_vars = [
        'N_FEWSHOT', 'N_VAL', 'N_TEST', 'BATCH_SIZE', 'CONCURRENCY',
        'ALPHA_GRID', 'CONF_GRID', 'STAGE2_THRESHOLD', 'SEED', 'VERSION',
        'MATRIX_VERSION', 'ACTIVE_MODEL', 'MODEL_NAME', 'QUANTIZATION',
        'MOE_BACKEND', 'KV_CACHE_DTYPE', 'GPU_MEM_FRAC', 'MAX_MODEL_LEN',
        'TENSOR_PARALLEL', 'DTYPE', 'THINKING_MODE', 'FORCE_REFRESH',
        'ENABLE_DOC_ENTITIES', 'DOC_NEIGHBOURS_K', 'N_NEIGHBOUR_POOL',
        'PER_GOAL_LAMBDA', 'CV_FOLDS_LAMBDA'
    ]
    notebook_config = {}
    for var in config_vars:
        if var in globals():
            notebook_config[var] = globals()[var]
        else:
            notebook_config[var] = "N/A"
    config_data['notebook_variables'] = notebook_config

    # 4. Data Fingerprints (from Cell 11)
    config_data['data_fingerprint_test_hash'] = globals().get('_hash_test', 'N/A')
    config_data['data_fingerprint_val_hash'] = globals().get('_hash_val', 'N/A')
    config_data['doc_methods_fingerprint_test_hash'] = globals().get('_doc_hash', 'N/A')

    # Save to file
    config_filename = os.path.join(results_dir, f'run_config_{version_string}.json')
    with open(config_filename, 'w') as f:
        json.dump(config_data, f, indent=4)
    print(f'Configuration saved to: {config_filename}')

save_config_for_reproducibility(RESULTS_DIR, VERSION)